# AuditStress — generation, text-aligned audit, and paper figures

This notebook contains the controlled-benchmark generator, the aligned QA1–QA3 runner, and publication outputs for metric compatibility version `audit-text-aligned-2.7`. Structural RDF defects are evaluated in QA1, predicate retention and bidirectional RDF–text support in QA2, and QA3 is restricted to language validity and legitimate multi-reference variation.

In [ ]:

from __future__ import annotations

from pathlib import Path
import hashlib
import importlib
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 180)


def find_repo_root() -> Path:
    override = Path("/home/vramon/notebooks/WebNLG_MT_eval")
    if override:
        root = Path(override).expanduser().resolve()
        if not (root / 'WebNLG_CA_BT').exists():
            raise FileNotFoundError(f'WebNLG_CA_BT not found under {root}')
        return root
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'WebNLG_CA_BT').exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Repository root not found. Set WEBNLG_REPO_ROOT or open the notebook inside the repository.'
    )


REPO_ROOT = find_repo_root()

# Use the same shared metric files as the corrected 01_QA1 notebook.
# Keeping one canonical copy prevents QA1.2 metric drift between the
# original-resource audit and AuditStress.
CORE_DIR = (
    REPO_ROOT
    / 'audit'
    / 'QA_evaluation'
    / 'audit_notebooks'
    / 'aligned'
)
CORE_PATH = CORE_DIR / 'audit_metric_core.py'
QA11_CORE_PATH = CORE_DIR / 'qa11_metric_core.py'

for required_path, label in (
    (CORE_PATH, 'Shared metric core'),
    (QA11_CORE_PATH, 'Shared QA1.1 metric core'),
):
    if not required_path.exists():
        raise FileNotFoundError(
            f'{label} not found: {required_path}. '
            'Copy it to audit/QA_evaluation/audit_notebooks/aligned/.'
        )

if str(CORE_DIR) not in sys.path:
    sys.path.insert(0, str(CORE_DIR))

import audit_metric_core as metric_core
import qa11_metric_core as qa11_core

importlib.reload(metric_core)
importlib.reload(qa11_core)


if metric_core.CORE_COMPATIBILITY != "audit-text-aligned-2.7":
    raise RuntimeError(
        "AuditStress requires audit_metric_core.py compatibility "
        "'audit-text-aligned-2.7', but found "
        f"{metric_core.CORE_COMPATIBILITY!r}."
    )

# Fail early if the shared RDF-field sanity rules no longer match the
# QA1.1 table.
qa11_core.assert_well_formedness_contract(
    metric_core.triple_well_formed
)

# Fail early if the shared recurring-component calculations drift from
# the QA1.2 table.
metric_core.assert_qa12_metric_contract()

# Fail early if semantic cosine values drift outside their theoretical range.
metric_core.assert_semantic_cosine_range_contract()

# Fail early if either shared QA2 implementation drifts from its table.
metric_core.assert_qa21_metric_contract()
metric_core.assert_qa22_metric_contract()

# Fail early if the shared QA3.1 implementation drifts from its table.
metric_core.assert_qa31_metric_contract()
metric_core.assert_qa32_metric_contract()

CORE_SHA256 = hashlib.sha256(CORE_PATH.read_bytes()).hexdigest()
QA11_CORE_SHA256 = hashlib.sha256(
    QA11_CORE_PATH.read_bytes()
).hexdigest()

print('Repository root:', REPO_ROOT)
print('Shared metric core:', CORE_PATH)
print('Metric compatibility:', metric_core.CORE_COMPATIBILITY)
print('Metric-core SHA-256:', CORE_SHA256)
print('Shared QA1.1 metric core:', QA11_CORE_PATH)
print('QA1.1 metric-core SHA-256:', QA11_CORE_SHA256)


## 0. Final-run configuration

In [ ]:

# Set these two switches to True for the first full run. Set them back to False
# when regenerating only tables and figures from existing CSV files.
RUN_GENERATION = True
RUN_AUDIT = True
RUN_VISUALISATIONS = True

# Benchmark and output names.
BENCHMARK_NAME = 'AuditStress_final'
AUDIT_NAME = f'{BENCHMARK_NAME}_audit'
BENCHMARK_DIR = REPO_ROOT / 'audit' / 'QA_evaluation' / 'results' / BENCHMARK_NAME
AUDIT_DIR = REPO_ROOT / 'audit' / 'QA_evaluation' / 'results' / AUDIT_NAME

# Final disjoint benchmark configuration.
BASE_SIZE = 1000
GENERATION_SEED = 2027
EXCLUDE_MANIFEST = REPO_ROOT / 'audit' / 'QA_evaluation' / 'results' / 'AuditStress_dev'
if not EXCLUDE_MANIFEST.exists():
    EXCLUDE_MANIFEST = None

CONSISTENCY_ITEMS = 10
CONSISTENCY_OCCURRENCES = 30
CONSISTENCY_CORRUPTION_FRACTION = 0.30
# QA1.2 recurrence threshold rho. It must remain at least 2.
QA12_MIN_RECURRING_OCCURRENCES = 2
BENIGN_CONSISTENCY_ITEMS = 5
MANUAL_SAMPLE = 300
WRITE_XML = True

# The obsolete generic addition is intentionally disabled.
COUNTS_PER_LANGUAGE = {
    'qa1_delete_triple': 250,
    'qa1_duplicate_triple': 250,
    'qa1_missing_text': 150,
    'qa1_wrong_lid': 150,
    'qa1_duplicate_lid': 150,
    'qa2_wrong_record_text': 400,
    'qa2_entity_substitution_triple': 400,
    'qa2_predicate_substitution_triple': 400,
    'qa2_entity_substitution_text': 300,
    'qa2_literal_text_only': 300,
    'qa2_literal_both': 300,
    'qa2_omit_one_sentence': 300,
    'qa2_omit_to_first_sentence': 300,
    'qa2_unsupported_addition': 0,
    'qa2_unsupported_addition_plausible': 300,
    'qa2_unsupported_addition_unrelated': 300,
    'qa3_full_english_copy': 300,
    'qa3_english_clause_insertion': 300,
    'qa2_structured_predicate_retention': 300,
    'qa1_placeholder_component': 250,
    'qa1_missing_component': 250,
    'qa1_source_markup': 250,
    'benign_alternative_reference': 400,
}

# Final audit configuration.
TRACKS = ['qa1', 'qa2', 'qa3']
SEMANTIC_BACKEND = 'sentence_transformer'  # final value
SEMANTIC_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE = 128
LANGUAGE_TRAINING_MAX = 6000
BOOTSTRAP_REPLICATES = 1000
FORCE_RECOMPUTE = False

# Paper-output directories.
PAPER_DIR = AUDIT_DIR / 'paper_outputs'
FIGURE_DIR = PAPER_DIR / 'figures'
TABLE_DIR = PAPER_DIR / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

assert COUNTS_PER_LANGUAGE['qa2_unsupported_addition'] == 0
assert COUNTS_PER_LANGUAGE['qa2_unsupported_addition_plausible'] > 0
assert COUNTS_PER_LANGUAGE['qa2_unsupported_addition_unrelated'] > 0
assert QA12_MIN_RECURRING_OCCURRENCES >= 2

print('Benchmark directory:', BENCHMARK_DIR)
print('Audit directory:', AUDIT_DIR)
print('Exclude manifest:', EXCLUDE_MANIFEST)
print('Generation enabled:', RUN_GENERATION)
print('Audit enabled:', RUN_AUDIT)
print('Visualisations enabled:', RUN_VISUALISATIONS)



# 1. Controlled benchmark generator

The following cells contain the complete version 1.2 generator. They create
paired clean/corrupted instances, nested omission and addition severities,
source-to-target literal corruptions, QA1 consistency groups, benign controls,
and disjoint held-out sets.


In [ ]:
from __future__ import annotations

import argparse
import copy
import hashlib
import html
import json
import math
import os
import random
import re
import unicodedata
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd


GENERATOR_VERSION = "1.3"

TARGET_LANGS = ("es", "ca")
LANG_CONFIG = {
    "es": {
        "label": "Spanish",
        "triple_set": "spanishtripleset",
        "triple_tag": "striple",
        "alt_suffix": " alternativo",
    },
    "ca": {
        "label": "Catalan",
        "triple_set": "catalantripleset",
        "triple_tag": "ctriple",
        "alt_suffix": " alternatiu",
    },
}


In [ ]:
TRACK_EXPECTATIONS: dict[str, dict[str, Any]] = {
    "clean": {
        "qa_track": "CLEAN", "qa_subqa": "control", "severity": "clean",
        "affected_side": "none", "expected_metrics": [],
        "expected_direction": "no change",
    },
    "qa1_delete_triple": {
        "qa_track": "QA1-STRUCT", "qa_subqa": "QA1.1", "severity": "mild",
        "affected_side": "target triples",
        "expected_metrics": ["target_triple_parity", "record_integrity"],
        "expected_direction": "fail",
    },
    "qa1_duplicate_triple": {
        "qa_track": "QA1-STRUCT", "qa_subqa": "QA1.1", "severity": "mild",
        "affected_side": "target triples",
        "expected_metrics": ["target_triple_parity", "record_integrity"],
        "expected_direction": "fail",
    },
    "qa1_missing_text": {
        "qa_track": "QA1-STRUCT", "qa_subqa": "QA1.1", "severity": "severe",
        "affected_side": "target text",
        "expected_metrics": ["target_text_present", "record_integrity"],
        "expected_direction": "fail",
    },
    "qa1_wrong_lid": {
        "qa_track": "QA1-STRUCT", "qa_subqa": "QA1.1", "severity": "mild",
        "affected_side": "metadata",
        "expected_metrics": ["lexicalisation_id_alignment", "record_integrity"],
        "expected_direction": "fail",
    },
    "qa1_duplicate_lid": {
        "qa_track": "QA1-STRUCT", "qa_subqa": "QA1.1", "severity": "mild",
        "affected_side": "metadata",
        "expected_metrics": ["lexicalisation_id_unique", "record_integrity"],
        "expected_direction": "fail",
    },
    "qa1_placeholder_component": {
        "qa_track": "QA1-STRUCT", "qa_subqa": "QA1.1", "severity": "severe",
        "affected_side": "target triples",
        "expected_metrics": ["target_rdf_well_formed", "record_integrity"],
        "expected_direction": "fail",
    },
    "qa1_missing_component": {
        "qa_track": "QA1-STRUCT", "qa_subqa": "QA1.1", "severity": "severe",
        "affected_side": "target triples",
        "expected_metrics": ["target_rdf_well_formed", "record_integrity"],
        "expected_direction": "fail",
    },
    "qa1_source_markup": {
        "qa_track": "QA1-STRUCT", "qa_subqa": "QA1.1", "severity": "mild",
        "affected_side": "target triples",
        "expected_metrics": ["target_rdf_well_formed", "record_integrity"],
        "expected_direction": "fail",
    },
    "qa1_predicate_inconsistency": {
        "qa_track": "QA1-CONSISTENCY", "qa_subqa": "QA1.2", "severity": "mild",
        "affected_side": "target triples",
        "expected_metrics": ["predicate_dominant_mapping_rate", "mapping_review_flag"],
        "expected_direction": "decrease/flag",
    },
    "qa1_entity_inconsistency": {
        "qa_track": "QA1-CONSISTENCY", "qa_subqa": "QA1.2", "severity": "mild",
        "affected_side": "target triples",
        "expected_metrics": ["entity_dominant_mapping_rate", "mapping_review_flag"],
        "expected_direction": "decrease/flag",
    },
    "qa1_consistency_clean_control": {
        "qa_track": "QA1-CONSISTENCY", "qa_subqa": "QA1.2", "severity": "clean",
        "affected_side": "none", "expected_metrics": ["dominant_mapping_rate"],
        "expected_direction": "stable",
    },
    "qa1_predicate_benign_variation": {
        "qa_track": "QA1-CONSISTENCY", "qa_subqa": "QA1.2", "severity": "benign",
        "affected_side": "none", "expected_metrics": ["mapping_review_flag"],
        "expected_direction": "review signal may fire; manual judgment should accept",
    },
    "qa1_entity_benign_variation": {
        "qa_track": "QA1-CONSISTENCY", "qa_subqa": "QA1.2", "severity": "benign",
        "affected_side": "none", "expected_metrics": ["mapping_review_flag"],
        "expected_direction": "review signal may fire; manual judgment should accept",
    },
    "qa2_wrong_record_text": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.1/QA2.2", "severity": "severe",
        "affected_side": "target text",
        "expected_metrics": ["text_similarity", "minimum_triple_coverage", "minimum_text_groundedness"],
        "expected_direction": "decrease",
    },
    "qa2_entity_substitution_triple": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.1/QA2.2", "severity": "mild",
        "affected_side": "target triples",
        "expected_metrics": ["triple_similarity", "minimum_triple_coverage"],
        "expected_direction": "decrease",
    },
    "qa2_predicate_substitution_triple": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.1", "severity": "mild",
        "affected_side": "target triples",
        "expected_metrics": ["predicate_similarity"],
        "expected_direction": "decrease",
    },
    "qa2_structured_predicate_retention": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.1", "severity": "mild",
        "affected_side": "target triples",
        "expected_metrics": ["structured_predicate_review_flag"],
        "expected_direction": "flag",
    },
    "qa2_entity_substitution_text": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.1/QA2.2", "severity": "mild",
        "affected_side": "target text",
        "expected_metrics": ["text_similarity", "minimum_triple_coverage"],
        "expected_direction": "decrease",
    },
    "qa2_literal_text_only": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.2", "severity": "mild",
        "affected_side": "target text",
        "expected_metrics": ["target_literal_retention"],
        "expected_direction": "decrease/fail",
    },
    "qa2_literal_both": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.1", "severity": "mild",
        "affected_side": "target triples and target text",
        "expected_metrics": ["source_target_literal_preservation"],
        "expected_direction": "decrease/fail",
    },
    "qa2_omit_one_sentence": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.2", "severity": "mild",
        "affected_side": "target text",
        "expected_metrics": ["minimum_triple_coverage"],
        "expected_direction": "decrease",
    },
    "qa2_omit_to_first_sentence": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.2", "severity": "severe",
        "affected_side": "target text",
        "expected_metrics": ["minimum_triple_coverage"],
        "expected_direction": "larger decrease than mild",
    },
    "qa2_unsupported_addition": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.2", "severity": "mild",
        "affected_side": "target text",
        "expected_metrics": ["minimum_text_groundedness"],
        "expected_direction": "decrease",
    },
    "qa2_unsupported_addition_plausible": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.2", "severity": "mild",
        "affected_side": "target text",
        "expected_metrics": ["minimum_text_groundedness"],
        "expected_direction": "decrease",
    },
    "qa2_unsupported_addition_unrelated": {
        "qa_track": "QA2-SEM", "qa_subqa": "QA2.2", "severity": "severe",
        "affected_side": "target text",
        "expected_metrics": ["minimum_text_groundedness"],
        "expected_direction": "larger decrease than plausible addition",
    },
    "qa3_full_english_copy": {
        "qa_track": "QA3-LANG", "qa_subqa": "QA3.1", "severity": "severe",
        "affected_side": "target text",
        "expected_metrics": ["whole_text_language", "exact_english_copy"],
        "expected_direction": "fail",
    },
    "qa3_english_clause_insertion": {
        "qa_track": "QA3-LANG", "qa_subqa": "QA3.1", "severity": "mild",
        "affected_side": "target text",
        "expected_metrics": ["code_switch", "target_language_probability"],
        "expected_direction": "flag/decrease",
    },
    "benign_alternative_reference": {
        "qa_track": "BENIGN-VAR", "qa_subqa": "QA3.2", "severity": "benign",
        "affected_side": "target text",
        "expected_metrics": ["aligned_reference_advantage", "expansion_ratio", "metric_non_decrease"],
        "expected_direction": "positive aligned advantage; hard language checks pass",
    },
}


In [ ]:

DEFAULT_COUNTS = {
    "qa1_delete_triple": 250,
    "qa1_duplicate_triple": 250,
    "qa1_missing_text": 150,
    "qa1_wrong_lid": 150,
    "qa1_duplicate_lid": 150,
    "qa2_wrong_record_text": 400,
    "qa2_entity_substitution_triple": 400,
    "qa2_predicate_substitution_triple": 400,
    "qa2_entity_substitution_text": 300,
    "qa2_literal_text_only": 300,
    "qa2_literal_both": 300,
    "qa2_omit_one_sentence": 300,
    "qa2_omit_to_first_sentence": 300,
    "qa2_unsupported_addition": 0,
    "qa2_unsupported_addition_plausible": 300,
    "qa2_unsupported_addition_unrelated": 300,
    "qa3_full_english_copy": 300,
    "qa3_english_clause_insertion": 300,
    "qa2_structured_predicate_retention": 300,
    "qa1_placeholder_component": 250,
    "qa1_missing_component": 250,
    "qa1_source_markup": 250,
    "benign_alternative_reference": 400,
}


@dataclass(frozen=True)
class BuildConfig:
    repo_root: str
    output_dir: str
    base_size: int = 1000
    seed: int = 42
    counts_per_language: Mapping[str, int] | None = None
    consistency_items_per_type: int = 10
    consistency_occurrences_per_item: int = 30
    consistency_corruption_fraction: float = 0.30
    benign_consistency_items_per_type: int = 5
    manual_verification_sample: int = 300
    exclude_manifest: str | None = None
    write_xml: bool = True


class NoisyBenchmarkError(RuntimeError):
    pass


def stable_seed(*parts: Any, base_seed: int = 42) -> int:
    payload = "||".join(map(str, parts)).encode("utf-8")
    digest = hashlib.sha256(payload).hexdigest()[:12]
    return (int(digest, 16) + int(base_seed)) % (2**32 - 1)


def normalize_space(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def normalize_label(value: Any) -> str:
    text = html.unescape(str(value or ""))
    text = re.sub(r"\^\^xsd:[A-Za-z]+\s*$", "", text)
    text = re.sub(r"@[A-Za-z-]+\s*$", "", text)
    text = text.strip(" \t\r\n\"'")
    text = text.replace("_", " ")
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    return text.casefold().strip()


def display_label(value: Any) -> str:
    return normalize_space(html.unescape(str(value or "")).replace("_", " ").strip(" \"'"))


def split_triple(value: Any) -> tuple[str, str, str]:
    parts = [part.strip() for part in str(value or "").split("|")]
    if len(parts) != 3:
        return "", "", ""
    return parts[0], parts[1], parts[2]


def join_triple(parts: Sequence[str]) -> str:
    return " | ".join(str(part).strip() for part in parts)


def is_probable_literal(value: str) -> bool:
    text = str(value or "").strip()
    return bool(
        re.search(r"\d", text)
        or re.search(r"\^\^xsd:", text)
        or re.match(r'^".*"(?:@[A-Za-z-]+)?$', text)
    )


def sentence_split(text: str) -> list[str]:
    text = normalize_space(text)
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÜÑÀÈÉÍÏÒÓÚÜÇ0-9])", text)
    return [part.strip() for part in parts if part.strip()]


def first_sentence(text: str) -> str:
    sentences = sentence_split(text)
    return sentences[0] if sentences else normalize_space(text)


def replace_case_insensitive_once(text: str, old: str, new: str) -> tuple[str, bool]:
    old_display = display_label(old)
    if not old_display:
        return text, False
    pattern = re.compile(re.escape(old_display), flags=re.IGNORECASE)
    updated, count = pattern.subn(new, text, count=1)
    return updated, bool(count)


def perturb_number_token(token: str) -> str:
    match = re.search(r"\d+", token)
    if not match:
        return token
    digits = match.group(0)
    value = int(digits)
    changed = value + 1 if value != 999999999 else value - 1
    replacement = str(changed).zfill(len(digits)) if len(str(changed)) <= len(digits) else str(changed)
    return token[: match.start()] + replacement + token[match.end() :]


def first_shared_numeric_token(text: str, triples: Sequence[str]) -> str | None:
    triple_text = " ".join(triples)
    tokens = re.findall(r"(?<!\w)\d+(?:[.,:/-]\d+)*(?!\w)", triple_text)
    for token in tokens:
        if token in text:
            return token
    return None


def list_xml_files(root: Path) -> tuple[list[Path], list[Path]]:
    usable: list[Path] = []
    ignored: list[Path] = []
    for path in root.rglob("*.xml"):
        rel = path.relative_to(root)
        if any(part.startswith(".") for part in rel.parts) or path.name.endswith("-checkpoint.xml"):
            ignored.append(path)
        else:
            usable.append(path)
    return sorted(usable), sorted(ignored)


def node_texts(entry: ET.Element, xpath: str) -> list[str]:
    return [normalize_space(node.text) for node in entry.findall(xpath)]


def lexicalisations_by_language(entry: ET.Element) -> dict[str, dict[str, str]]:
    result: dict[str, dict[str, str]] = defaultdict(dict)
    for lex in entry.findall("lex"):
        lang = normalize_space(lex.get("lang"))
        lid = normalize_space(lex.get("lid"))
        if lang and lid:
            result[lang][lid] = normalize_space(lex.text)
    return dict(result)


def parse_repository(repo_root: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    data_root = repo_root / "WebNLG_CA_BT"
    if not data_root.exists():
        raise FileNotFoundError(f"WebNLG_CA_BT was not found under {repo_root}")

    xml_files, ignored = list_xml_files(data_root)
    records: list[dict[str, Any]] = []
    parse_errors: list[dict[str, str]] = []

    for xml_file in xml_files:
        rel = xml_file.relative_to(data_root)
        split = rel.parts[0] if rel.parts else "unknown"
        bucket = rel.parts[1] if len(rel.parts) > 1 else "unknown"
        try:
            tree = ET.parse(xml_file)
        except ET.ParseError as exc:
            parse_errors.append({"xml_path": rel.as_posix(), "error": str(exc)})
            continue

        for entry in tree.findall(".//entry"):
            source_triples = node_texts(entry, ".//modifiedtripleset/mtriple")
            es_triples = node_texts(entry, ".//spanishtripleset/striple")
            ca_triples = node_texts(entry, ".//catalantripleset/ctriple")
            enbt_triples = node_texts(entry, ".//enbttripleset/bttriple") or list(source_triples)
            lex = lexicalisations_by_language(entry)
            shared_lids = sorted(set(lex.get("en", {})) & set(lex.get("es", {})) & set(lex.get("ca", {})))
            if not source_triples or not es_triples or not ca_triples or not shared_lids:
                continue
            declared_size_raw = normalize_space(entry.get("size"))
            try:
                declared_size = int(declared_size_raw)
                declared_size_valid = declared_size >= 0
            except (TypeError, ValueError):
                declared_size = None
                declared_size_valid = False

            expected_triple_count = (
                declared_size
                if declared_size_valid
                else len(source_triples)
            )

            record_key = f"{rel.as_posix()}::{entry.get('eid', '')}"
            records.append({
                "record_key": record_key,
                "xml_path": rel.as_posix(),
                "split": split,
                "triple_bucket": bucket,
                "category": normalize_space(entry.get("category")),
                "eid": normalize_space(entry.get("eid")),
                "shape": normalize_space(entry.get("shape")),
                "shape_type": normalize_space(entry.get("shape_type")),
                "declared_size_raw": declared_size_raw,
                "declared_size": declared_size,
                "declared_size_valid": declared_size_valid,
                "expected_triple_count": expected_triple_count,
                "source_triples": source_triples,
                "es_triples": es_triples,
                "ca_triples": ca_triples,
                "enbt_triples": enbt_triples,
                "lex_en": lex.get("en", {}),
                "lex_es": lex.get("es", {}),
                "lex_ca": lex.get("ca", {}),
                "lex_en_bt": lex.get("en_bt", {}),
                "shared_lids": shared_lids,
            })

    records_df = pd.DataFrame(records)
    if records_df.empty:
        raise NoisyBenchmarkError("No aligned English/Spanish/Catalan records were parsed.")

    return (
        records_df,
        pd.DataFrame({"xml_path": [p.relative_to(data_root).as_posix() for p in ignored]}),
        pd.DataFrame(parse_errors),
    )


In [ ]:


def one_lexicalisation_per_record(records: pd.DataFrame, seed: int) -> pd.DataFrame:
    rows = []
    for row in records.itertuples(index=False):
        rng = random.Random(stable_seed(row.record_key, "base_lid", base_seed=seed))
        lid = rng.choice(list(row.shared_lids))
        rows.append(record_to_base_row(row, lid))
    return pd.DataFrame(rows)


def record_to_base_row(row: Any, lid: str) -> dict[str, Any]:
    enbt_text = row.lex_en_bt.get(lid, row.lex_en.get(lid, ""))
    return {
        "base_pair_key": f"{row.record_key}::{lid}",
        "record_key": row.record_key,
        "xml_path": row.xml_path,
        "split": row.split,
        "triple_bucket": row.triple_bucket,
        "category": row.category,
        "source_eid": row.eid,
        "source_lid": lid,
        "shape": row.shape,
        "shape_type": row.shape_type,
        "declared_size_raw": row.declared_size_raw,
        "declared_size": row.declared_size,
        "declared_size_valid": bool(row.declared_size_valid),
        "expected_triple_count": int(row.expected_triple_count),
        "source_triples": list(row.source_triples),
        "es_triples": list(row.es_triples),
        "ca_triples": list(row.ca_triples),
        "enbt_triples": list(row.enbt_triples),
        "text_en": row.lex_en.get(lid, ""),
        "text_es": row.lex_es.get(lid, ""),
        "text_ca": row.lex_ca.get(lid, ""),
        "text_en_bt": enbt_text,
        "all_lex_en": dict(row.lex_en),
        "all_lex_es": dict(row.lex_es),
        "all_lex_ca": dict(row.lex_ca),
    }


def proportional_stratified_sample(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if n >= len(df):
        return df.sample(frac=1, random_state=seed).reset_index(drop=True)
    groups = list(df.groupby(["split", "expected_triple_count"], dropna=False))
    sizes = np.array([len(group) for _, group in groups], dtype=float)
    raw = sizes / sizes.sum() * n
    quotas = np.floor(raw).astype(int)
    quotas = np.minimum(quotas, sizes.astype(int))
    remainder = n - quotas.sum()
    fractional_order = np.argsort(-(raw - quotas))
    for idx in fractional_order:
        if remainder <= 0:
            break
        if quotas[idx] < sizes[idx]:
            quotas[idx] += 1
            remainder -= 1
    pieces = []
    for (key, group), quota in zip(groups, quotas):
        if quota <= 0:
            continue
        pieces.append(group.sample(n=int(quota), random_state=stable_seed(key, base_seed=seed)))
    result = pd.concat(pieces, ignore_index=True)
    if len(result) < n:
        remaining = df[~df["base_pair_key"].isin(result["base_pair_key"])]
        extra = remaining.sample(n=n - len(result), random_state=seed + 1)
        result = pd.concat([result, extra], ignore_index=True)
    return result.sample(frac=1, random_state=seed).reset_index(drop=True)


def make_variant_id(base_pair_key: str, language: str, corruption_type: str, ordinal: int = 0) -> str:
    digest = hashlib.sha1(
        f"{base_pair_key}|{language}|{corruption_type}|{ordinal}".encode("utf-8")
    ).hexdigest()[:12]
    return f"{corruption_type}__{language}__{digest}"


def base_variant(
    row: Mapping[str, Any],
    language: str,
    corruption_type: str,
    ordinal: int = 0,
) -> dict[str, Any]:
    """Create one controlled variant while preserving entry-level QA1.1 data.

    QA2 and QA3 operate on the selected lexicalisation stored in the scalar
    text/LID fields. QA1.1 receives all lexicalisation identifiers and target
    texts from the RDF entry through the list-valued fields.
    """
    expectation = TRACK_EXPECTATIONS[corruption_type]
    target_triples = list(row[f"{language}_triples"])
    target_text = row[f"text_{language}"]

    english_lexicalisations = dict(row["all_lex_en"])
    target_lexicalisations = dict(row[f"all_lex_{language}"])
    source_lids = list(english_lexicalisations.keys())
    target_lids = list(target_lexicalisations.keys())
    target_texts = [
        target_lexicalisations[lid]
        for lid in target_lids
    ]

    selected_lid = row["source_lid"]
    if selected_lid not in target_lids:
        raise NoisyBenchmarkError(
            f"Selected LID {selected_lid!r} is missing from the "
            f"{language} lexicalisations for {row['record_key']}"
        )

    return {
        "variant_id": make_variant_id(
            row["base_pair_key"],
            language,
            corruption_type,
            ordinal,
        ),
        "base_pair_key": row["base_pair_key"],
        "record_key": row["record_key"],
        "xml_path": row["xml_path"],
        "split": row["split"],
        "triple_bucket": row["triple_bucket"],
        "category": row["category"],
        "source_eid": row["source_eid"],
        "source_lid": selected_lid,
        "variant_lid": selected_lid,
        "source_lids": source_lids,
        "variant_lids": target_lids,
        "shape": row["shape"],
        "shape_type": row["shape_type"],
        "declared_size_raw": row["declared_size_raw"],
        "declared_size": row["declared_size"],
        "declared_size_valid": bool(row["declared_size_valid"]),
        "expected_triple_count": int(row["expected_triple_count"]),
        "language": language,
        "language_label": LANG_CONFIG[language]["label"],
        "qa_track": expectation["qa_track"],
        "qa_subqa": expectation["qa_subqa"],
        "corruption_type": corruption_type,
        "severity": expectation["severity"],
        "affected_side": expectation["affected_side"],
        "expected_metrics": list(expectation["expected_metrics"]),
        "expected_direction": expectation["expected_direction"],
        "is_corrupted": corruption_type not in {
            "clean",
            "qa1_consistency_clean_control",
            "benign_alternative_reference",
            "qa1_predicate_benign_variation",
            "qa1_entity_benign_variation",
        },
        "is_benign": corruption_type in {
            "benign_alternative_reference",
            "qa1_predicate_benign_variation",
            "qa1_entity_benign_variation",
        },
        "source_triples": list(row["source_triples"]),
        "clean_target_triples": target_triples,
        "variant_target_triples": list(target_triples),
        "other_language_triples": list(
            row[f"{'ca' if language == 'es' else 'es'}_triples"]
        ),
        "enbt_triples": list(row["enbt_triples"]),
        "source_text": row["text_en"],
        "clean_target_text": target_text,
        "variant_target_text": target_text,
        "variant_target_texts": target_texts,
        "other_language_text": row[
            f"text_{'ca' if language == 'es' else 'es'}"
        ],
        "enbt_text": row["text_en_bt"],
        "donor_record_key": "",
        "donor_value": "",
        "changed_value_before": "",
        "changed_value_after": "",
        "consistency_group_id": "",
        "xml_duplicate_target_lex": False,
        "generation_notes": "",
        "perturbation_pair_id": "",
        "original_sentence_count": np.nan,
        "remaining_sentence_count": np.nan,
        "removed_sentence_count": np.nan,
        "removed_token_fraction": np.nan,
        "alternative_reference_lid": "",
        "alternative_source_text": "",
    }


def sample_eligible(df: pd.DataFrame, n: int, seed: int, mask: pd.Series | None = None) -> pd.DataFrame:
    eligible = df if mask is None else df.loc[mask]
    if eligible.empty:
        return eligible.copy()
    return eligible.sample(n=min(n, len(eligible)), random_state=seed).reset_index(drop=True)


def donor_row(pool: pd.DataFrame, base: Mapping[str, Any], seed: int, same_size: bool = True) -> Mapping[str, Any]:
    candidates = pool[pool["record_key"] != base["record_key"]]
    if same_size:
        matched = candidates[
            candidates["expected_triple_count"]
            == base["expected_triple_count"]
        ]
        if not matched.empty:
            candidates = matched
    different_category = candidates[candidates["category"] != base["category"]]
    if not different_category.empty:
        candidates = different_category
    if candidates.empty:
        raise NoisyBenchmarkError("No donor record was available.")
    return candidates.sample(n=1, random_state=seed).iloc[0].to_dict()


def plausible_unsupported_addition(triples: Sequence[str], language: str) -> str:
    entity = ""
    for triple in triples:
        subject, _, obj = split_triple(triple)
        for candidate in (subject, obj):
            if candidate and not is_probable_literal(candidate):
                entity = display_label(candidate)
                break
        if entity:
            break
    entity = entity or ("L'entitat" if language == "ca" else "La entidad")
    if language == "ca":
        return f"{entity} també va rebre un reconeixement internacional."
    return f"{entity} también recibió un reconocimiento internacional."


def load_excluded_record_keys(value: str | None) -> set[str]:
    if not value:
        return set()
    path = Path(value).expanduser().resolve()
    if path.is_dir():
        pkl = path / "noisy_manifest.pkl"
        csv = path / "noisy_manifest.csv"
        path = pkl if pkl.exists() else csv
    if not path.exists():
        raise FileNotFoundError(f"Excluded manifest was not found: {path}")
    frame = pd.read_pickle(path) if path.suffix == ".pkl" else pd.read_csv(path, usecols=["record_key"])
    if "record_key" not in frame.columns:
        raise NoisyBenchmarkError(f"Excluded manifest has no record_key column: {path}")
    return set(frame["record_key"].dropna().astype(str))


def entity_pool(base_df: pd.DataFrame, language: str) -> list[str]:
    values: set[str] = set()
    for triples in base_df[f"{language}_triples"]:
        for triple in triples:
            subject, _, obj = split_triple(triple)
            for value in (subject, obj):
                if value and not is_probable_literal(value):
                    values.add(value)
    return sorted(values)


def predicate_pool(base_df: pd.DataFrame, language: str) -> list[str]:
    values = {
        split_triple(triple)[1]
        for triples in base_df[f"{language}_triples"]
        for triple in triples
        if split_triple(triple)[1]
    }
    return sorted(values)


def choose_different(values: Sequence[str], current: str, seed: int) -> str:
    candidates = [value for value in values if normalize_label(value) != normalize_label(current)]
    if not candidates:
        raise NoisyBenchmarkError("No distinct replacement value was available.")
    return random.Random(seed).choice(candidates)


In [ ]:


def mutate_triple_component(
    triples: Sequence[str],
    triple_index: int,
    role_index: int,
    new_value: str,
) -> tuple[list[str], str]:
    updated = list(triples)
    parts = list(split_triple(updated[triple_index]))
    old = parts[role_index]
    parts[role_index] = new_value
    updated[triple_index] = join_triple(parts)
    return updated, old


In [ ]:


def generate_standard_variants(
    base_df: pd.DataFrame,
    counts: Mapping[str, int],
    seed: int,
) -> list[dict[str, Any]]:
    variants: list[dict[str, Any]] = []
    entity_pools = {lang: entity_pool(base_df, lang) for lang in TARGET_LANGS}
    predicate_pools = {lang: predicate_pool(base_df, lang) for lang in TARGET_LANGS}

    for language in TARGET_LANGS:
        for corruption_type, requested_n in counts.items():
            # Mild and severe omissions are generated together from the same
            # base instances by generate_nested_omission_variants().
            if corruption_type in {
                "qa2_omit_one_sentence",
                "qa2_omit_to_first_sentence",
                "qa2_unsupported_addition_plausible",
                "qa2_unsupported_addition_unrelated",
            }:
                continue
            n = int(requested_n)
            if n <= 0:
                continue
            type_seed = stable_seed(language, corruption_type, base_seed=seed)

            if corruption_type in {"qa1_delete_triple", "qa2_omit_one_sentence", "qa2_omit_to_first_sentence"}:
                if corruption_type == "qa1_delete_triple":
                    mask = base_df["expected_triple_count"] >= 2
                else:
                    mask = base_df[f"text_{language}"].map(lambda text: len(sentence_split(text)) >= 2)
            elif corruption_type in {"qa2_literal_text_only", "qa2_literal_both"}:
                mask = base_df.apply(
                    lambda row: first_shared_numeric_token(row[f"text_{language}"], row[f"{language}_triples"]) is not None,
                    axis=1,
                )
            elif corruption_type == "qa2_entity_substitution_text":
                def has_surface_entity(row: pd.Series) -> bool:
                    text_norm = normalize_label(row[f"text_{language}"])
                    for triple in row[f"{language}_triples"]:
                        subject, _, obj = split_triple(triple)
                        for value in (subject, obj):
                            label = normalize_label(value)
                            if label and not is_probable_literal(value) and label in text_norm:
                                return True
                    return False
                mask = base_df.apply(has_surface_entity, axis=1)
            elif corruption_type == "benign_alternative_reference":
                def has_distinct_alternative(row: pd.Series) -> bool:
                    clean_text = normalize_space(row[f"text_{language}"])
                    return any(
                        lid != row["source_lid"]
                        and normalize_space(alt_text)
                        and normalize_space(alt_text) != clean_text
                        for lid, alt_text in row[f"all_lex_{language}"].items()
                    )

                mask = base_df.apply(has_distinct_alternative, axis=1)
            else:
                mask = None

            sample = sample_eligible(base_df, n, type_seed, mask)
            for ordinal, base_row in enumerate(sample.to_dict("records")):
                variant = base_variant(base_row, language, corruption_type, ordinal)
                rng_seed = stable_seed(variant["variant_id"], base_seed=seed)
                rng = random.Random(rng_seed)
                triples = list(variant["clean_target_triples"])
                text = variant["clean_target_text"]

                if corruption_type == "qa1_delete_triple":
                    idx = rng.randrange(len(triples))
                    removed = triples.pop(idx)
                    variant["variant_target_triples"] = triples
                    variant["changed_value_before"] = removed
                    variant["changed_value_after"] = "<deleted>"

                elif corruption_type == "qa1_duplicate_triple":
                    idx = rng.randrange(len(triples))
                    triples.insert(idx + 1, triples[idx])
                    variant["variant_target_triples"] = triples
                    variant["changed_value_before"] = triples[idx]
                    variant["changed_value_after"] = f"{triples[idx]} <duplicated>"

                elif corruption_type == "qa1_missing_text":
                    lid = variant["source_lid"]
                    lid_index = variant["variant_lids"].index(lid)
                    variant["variant_target_text"] = ""
                    variant["variant_target_texts"][lid_index] = ""
                    variant["changed_value_before"] = text
                    variant["changed_value_after"] = "<empty>"

                elif corruption_type == "qa1_wrong_lid":
                    original_lid = variant["source_lid"]
                    lid_index = variant["variant_lids"].index(original_lid)
                    wrong_lid = f"{original_lid}_NOISE"
                    variant["variant_lid"] = wrong_lid
                    variant["variant_lids"][lid_index] = wrong_lid
                    variant["changed_value_before"] = original_lid
                    variant["changed_value_after"] = wrong_lid

                elif corruption_type == "qa1_duplicate_lid":
                    lid = variant["source_lid"]
                    lid_index = variant["variant_lids"].index(lid)
                    duplicate_text = variant["variant_target_texts"][lid_index]
                    variant["variant_lids"].append(lid)
                    variant["variant_target_texts"].append(duplicate_text)
                    # Retained only for XML serialisation compatibility;
                    # the metric itself counts the actual identifier list.
                    variant["xml_duplicate_target_lex"] = True
                    variant["changed_value_before"] = lid
                    variant["changed_value_after"] = f"{lid} <duplicated>"

                elif corruption_type == "qa2_wrong_record_text":
                    donor = donor_row(base_df, base_row, rng_seed, same_size=True)
                    donor_text = donor[f"text_{language}"]
                    variant["variant_target_text"] = donor_text
                    variant["donor_record_key"] = donor["record_key"]
                    variant["donor_value"] = donor_text
                    variant["changed_value_before"] = text
                    variant["changed_value_after"] = donor_text

                elif corruption_type == "qa2_entity_substitution_triple":
                    choices = []
                    for ti, triple in enumerate(triples):
                        s, _, o = split_triple(triple)
                        if s and not is_probable_literal(s):
                            choices.append((ti, 0, s))
                        if o and not is_probable_literal(o):
                            choices.append((ti, 2, o))
                    if not choices:
                        continue
                    ti, role, old = rng.choice(choices)
                    new = choose_different(entity_pools[language], old, rng_seed)
                    updated, _ = mutate_triple_component(triples, ti, role, new)
                    variant["variant_target_triples"] = updated
                    variant["donor_value"] = new
                    variant["changed_value_before"] = old
                    variant["changed_value_after"] = new

                elif corruption_type == "qa2_predicate_substitution_triple":
                    ti = rng.randrange(len(triples))
                    old = split_triple(triples[ti])[1]
                    new = choose_different(predicate_pools[language], old, rng_seed)
                    updated, _ = mutate_triple_component(triples, ti, 1, new)
                    variant["variant_target_triples"] = updated
                    variant["donor_value"] = new
                    variant["changed_value_before"] = old
                    variant["changed_value_after"] = new

                elif corruption_type == "qa2_entity_substitution_text":
                    text_norm = normalize_label(text)
                    choices = []
                    for triple in triples:
                        subject, _, obj = split_triple(triple)
                        for value in (subject, obj):
                            label = normalize_label(value)
                            if label and not is_probable_literal(value) and label in text_norm:
                                choices.append(value)
                    if not choices:
                        continue
                    old = rng.choice(choices)
                    new_raw = choose_different(entity_pools[language], old, rng_seed)
                    new = display_label(new_raw)
                    updated_text, changed = replace_case_insensitive_once(text, old, new)
                    if not changed:
                        continue
                    variant["variant_target_text"] = updated_text
                    variant["donor_value"] = new_raw
                    variant["changed_value_before"] = display_label(old)
                    variant["changed_value_after"] = new

                elif corruption_type in {"qa2_literal_text_only", "qa2_literal_both"}:
                    token = first_shared_numeric_token(text, triples)
                    if token is None:
                        continue
                    changed = perturb_number_token(token)
                    variant["variant_target_text"] = text.replace(token, changed, 1)
                    if corruption_type == "qa2_literal_both":
                        variant["variant_target_triples"] = [
                            triple.replace(token, changed) for triple in triples
                        ]
                    variant["changed_value_before"] = token
                    variant["changed_value_after"] = changed

                elif corruption_type == "qa2_omit_one_sentence":
                    sentences = sentence_split(text)
                    if len(sentences) < 2:
                        continue
                    removed_index = rng.randrange(len(sentences))
                    removed = sentences.pop(removed_index)
                    variant["variant_target_text"] = " ".join(sentences)
                    variant["changed_value_before"] = removed
                    variant["changed_value_after"] = "<deleted sentence>"

                elif corruption_type == "qa2_omit_to_first_sentence":
                    sentences = sentence_split(text)
                    if len(sentences) < 2:
                        continue
                    variant["variant_target_text"] = sentences[0]
                    variant["changed_value_before"] = " ".join(sentences[1:])
                    variant["changed_value_after"] = "<all later sentences deleted>"

                elif corruption_type in {"qa2_unsupported_addition", "qa2_unsupported_addition_unrelated"}:
                    donor = donor_row(base_df, base_row, rng_seed, same_size=False)
                    addition = first_sentence(donor[f"text_{language}"])
                    variant["variant_target_text"] = f"{text} {addition}".strip()
                    variant["donor_record_key"] = donor["record_key"]
                    variant["donor_value"] = addition
                    variant["changed_value_before"] = "<none>"
                    variant["changed_value_after"] = addition
                    variant["generation_notes"] = "Severe unrelated unsupported addition from another record."

                elif corruption_type == "qa2_unsupported_addition_plausible":
                    addition = plausible_unsupported_addition(triples, language)
                    variant["variant_target_text"] = f"{text} {addition}".strip()
                    variant["donor_value"] = addition
                    variant["changed_value_before"] = "<none>"
                    variant["changed_value_after"] = addition
                    variant["generation_notes"] = "Mild fluent unsupported addition involving an existing entity."

                elif corruption_type == "qa3_full_english_copy":
                    variant["variant_target_text"] = variant["source_text"]
                    variant["changed_value_before"] = text
                    variant["changed_value_after"] = variant["source_text"]

                elif corruption_type == "qa3_english_clause_insertion":
                    clause = first_sentence(variant["source_text"])
                    variant["variant_target_text"] = f"{text} {clause}".strip()
                    variant["changed_value_before"] = "<none>"
                    variant["changed_value_after"] = clause

                elif corruption_type == "qa2_structured_predicate_retention":
                    ti = rng.randrange(min(len(triples), len(variant["source_triples"])))
                    source_predicate = split_triple(variant["source_triples"][ti])[1]
                    old = split_triple(triples[ti])[1]
                    updated, _ = mutate_triple_component(triples, ti, 1, source_predicate)
                    variant["variant_target_triples"] = updated
                    variant["changed_value_before"] = old
                    variant["changed_value_after"] = source_predicate

                elif corruption_type == "qa1_placeholder_component":
                    ti = rng.randrange(len(triples))
                    updated, old = mutate_triple_component(triples, ti, 2, "NULL")
                    variant["variant_target_triples"] = updated
                    variant["changed_value_before"] = old
                    variant["changed_value_after"] = "NULL"

                elif corruption_type == "qa1_missing_component":
                    ti = rng.randrange(len(triples))
                    role = rng.choice([0, 1, 2])
                    updated, old = mutate_triple_component(triples, ti, role, "")
                    variant["variant_target_triples"] = updated
                    variant["changed_value_before"] = old
                    variant["changed_value_after"] = "<empty>"

                elif corruption_type == "qa1_source_markup":
                    ti = rng.randrange(len(triples))
                    role = rng.choice([0, 1, 2])
                    old = split_triple(triples[ti])[role]
                    new = f"{old}@en"
                    updated, _ = mutate_triple_component(triples, ti, role, new)
                    variant["variant_target_triples"] = updated
                    variant["changed_value_before"] = old
                    variant["changed_value_after"] = new

                elif corruption_type == "benign_alternative_reference":
                    clean_text_normalized = normalize_space(text)
                    alternatives = {
                        lid: alt_text
                        for lid, alt_text in base_row[f"all_lex_{language}"].items()
                        if (
                            lid != base_row["source_lid"]
                            and normalize_space(alt_text)
                            and normalize_space(alt_text) != clean_text_normalized
                        )
                    }
                    if not alternatives:
                        continue
                    # Prefer the naturally occurring reference with the greatest surface divergence.
                    def surface_ratio(item: tuple[str, str]) -> float:
                        from difflib import SequenceMatcher
                        return SequenceMatcher(None, normalize_label(text), normalize_label(item[1])).ratio()
                    alt_lid, alt_text = min(alternatives.items(), key=surface_ratio)
                    variant["variant_target_text"] = alt_text
                    variant["donor_value"] = alt_lid
                    variant["alternative_reference_lid"] = alt_lid
                    variant["alternative_source_text"] = base_row["all_lex_en"].get(alt_lid, "")
                    variant["changed_value_before"] = text
                    variant["changed_value_after"] = alt_text
                    variant["generation_notes"] = "Naturally occurring alternative lexicalisation for the same RDF entry."

                else:
                    raise NoisyBenchmarkError(f"Unsupported corruption type: {corruption_type}")

                if (
                    variant["variant_target_triples"] == variant["clean_target_triples"]
                    and variant["variant_target_text"] == variant["clean_target_text"]
                    and variant["variant_lid"] == variant["source_lid"]
                    and not variant["xml_duplicate_target_lex"]
                    and not variant["is_benign"]
                ):
                    continue
                variants.append(variant)
    return variants


In [ ]:


def generate_nested_omission_variants(
    base_df: pd.DataFrame,
    counts: Mapping[str, int],
    seed: int,
) -> list[dict[str, Any]]:
    variants: list[dict[str, Any]] = []
    mild_n = int(counts.get("qa2_omit_one_sentence", 0))
    severe_n = int(counts.get("qa2_omit_to_first_sentence", 0))
    requested = max(mild_n, severe_n)
    if requested <= 0:
        return variants

    for language in TARGET_LANGS:
        mask = base_df[f"text_{language}"].map(lambda text: len(sentence_split(text)) >= 3)
        sample = sample_eligible(
            base_df,
            requested,
            stable_seed(language, "nested-omission", base_seed=seed),
            mask,
        )
        for ordinal, base_row in enumerate(sample.to_dict("records")):
            text = base_row[f"text_{language}"]
            sentences = sentence_split(text)
            if len(sentences) < 3:
                continue
            pair_id = f"omission::{language}::{base_row['base_pair_key']}"
            rng = random.Random(stable_seed(pair_id, base_seed=seed))
            removable = list(range(1, len(sentences)))
            mild_removed_index = rng.choice(removable)

            if ordinal < mild_n:
                mild = base_variant(base_row, language, "qa2_omit_one_sentence", ordinal)
                mild_sentences = list(sentences)
                removed = mild_sentences.pop(mild_removed_index)
                mild_text = " ".join(mild_sentences)
                mild["variant_target_text"] = mild_text
                mild["changed_value_before"] = removed
                mild["changed_value_after"] = "<deleted sentence>"
                mild["perturbation_pair_id"] = pair_id
                mild["original_sentence_count"] = len(sentences)
                mild["remaining_sentence_count"] = len(mild_sentences)
                mild["removed_sentence_count"] = 1
                mild["removed_token_fraction"] = 1 - (
                    len(mild_text.split()) / max(1, len(text.split()))
                )
                mild["generation_notes"] = "Nested mild omission; one non-initial sentence removed."
                variants.append(mild)

            if ordinal < severe_n:
                severe = base_variant(base_row, language, "qa2_omit_to_first_sentence", ordinal)
                severe_text = sentences[0]
                severe["variant_target_text"] = severe_text
                severe["changed_value_before"] = " ".join(sentences[1:])
                severe["changed_value_after"] = "<all non-initial sentences deleted>"
                severe["perturbation_pair_id"] = pair_id
                severe["original_sentence_count"] = len(sentences)
                severe["remaining_sentence_count"] = 1
                severe["removed_sentence_count"] = len(sentences) - 1
                severe["removed_token_fraction"] = 1 - (
                    len(severe_text.split()) / max(1, len(text.split()))
                )
                severe["generation_notes"] = "Nested severe omission; only the first sentence retained."
                variants.append(severe)
    return variants


def generate_paired_addition_variants(
    base_df: pd.DataFrame,
    counts: Mapping[str, int],
    seed: int,
) -> list[dict[str, Any]]:
    variants: list[dict[str, Any]] = []
    plausible_n = int(counts.get("qa2_unsupported_addition_plausible", 0))
    unrelated_n = int(counts.get("qa2_unsupported_addition_unrelated", 0))
    requested = max(plausible_n, unrelated_n)
    if requested <= 0:
        return variants

    for language in TARGET_LANGS:
        sample = sample_eligible(
            base_df,
            requested,
            stable_seed(language, "paired-addition", base_seed=seed),
        )
        for ordinal, base_row in enumerate(sample.to_dict("records")):
            text = base_row[f"text_{language}"]
            triples = list(base_row[f"{language}_triples"])
            pair_id = f"addition::{language}::{base_row['base_pair_key']}"
            rng_seed = stable_seed(pair_id, base_seed=seed)

            if ordinal < plausible_n:
                plausible = base_variant(
                    base_row, language, "qa2_unsupported_addition_plausible", ordinal
                )
                addition = plausible_unsupported_addition(triples, language)
                plausible["variant_target_text"] = f"{text} {addition}".strip()
                plausible["donor_value"] = addition
                plausible["changed_value_before"] = "<none>"
                plausible["changed_value_after"] = addition
                plausible["perturbation_pair_id"] = pair_id
                plausible["generation_notes"] = (
                    "Paired mild fluent unsupported addition involving an existing entity."
                )
                variants.append(plausible)

            if ordinal < unrelated_n:
                unrelated = base_variant(
                    base_row, language, "qa2_unsupported_addition_unrelated", ordinal
                )
                donor = donor_row(base_df, base_row, rng_seed, same_size=False)
                addition = first_sentence(donor[f"text_{language}"])
                unrelated["variant_target_text"] = f"{text} {addition}".strip()
                unrelated["donor_record_key"] = donor["record_key"]
                unrelated["donor_value"] = addition
                unrelated["changed_value_before"] = "<none>"
                unrelated["changed_value_after"] = addition
                unrelated["perturbation_pair_id"] = pair_id
                unrelated["generation_notes"] = (
                    "Paired severe unrelated unsupported addition from another record."
                )
                variants.append(unrelated)
    return variants


def component_occurrence_table(records: pd.DataFrame, component_type: str) -> pd.DataFrame:
    rows = []
    for row in records.itertuples(index=False):
        for triple_index, source_triple in enumerate(row.source_triples):
            source_s, source_p, source_o = split_triple(source_triple)
            for language in TARGET_LANGS:
                target_triples = getattr(row, f"{language}_triples")
                if triple_index >= len(target_triples):
                    continue
                target_s, target_p, target_o = split_triple(target_triples[triple_index])
                if component_type == "predicate":
                    source_value, target_value, role_index = source_p, target_p, 1
                    values = [(source_value, target_value, role_index)]
                else:
                    values = [
                        (source_s, target_s, 0),
                        (source_o, target_o, 2),
                    ]
                for source_value, target_value, role_index in values:
                    if not source_value or not target_value:
                        continue
                    if component_type == "entity" and is_probable_literal(source_value):
                        continue
                    rows.append({
                        "record_key": row.record_key,
                        "language": language,
                        "source_norm": normalize_label(source_value),
                        "source_value": source_value,
                        "target_value": target_value,
                        "triple_index": triple_index,
                        "role_index": role_index,
                    })
    return pd.DataFrame(rows)


In [ ]:


def generate_consistency_track(
    records_df: pd.DataFrame,
    items_per_type: int,
    occurrences_per_item: int,
    corruption_fraction: float,
    seed: int,
) -> list[dict[str, Any]]:
    variants: list[dict[str, Any]] = []
    record_lookup = records_df.set_index("record_key", drop=False)

    for component_type in ("predicate", "entity"):
        occurrences = component_occurrence_table(records_df, component_type)
        if occurrences.empty:
            continue
        for language in TARGET_LANGS:
            lang_occ = occurrences[occurrences["language"] == language]
            counts = lang_occ.groupby("source_norm")["record_key"].nunique().sort_values(ascending=False)
            eligible_items = counts[counts >= max(5, occurrences_per_item)].head(items_per_type).index.tolist()
            for item_index, source_norm in enumerate(eligible_items):
                item_occ = lang_occ[lang_occ["source_norm"] == source_norm].drop_duplicates("record_key")
                item_occ = item_occ.sample(
                    n=min(occurrences_per_item, len(item_occ)),
                    random_state=stable_seed(component_type, language, source_norm, base_seed=seed),
                ).reset_index(drop=True)
                n_corrupt = max(1, round(len(item_occ) * corruption_fraction))
                corrupted_indices = set(
                    item_occ.sample(
                        n=n_corrupt,
                        random_state=stable_seed("corrupt", component_type, language, source_norm, base_seed=seed),
                    ).index.tolist()
                )
                group_id = f"{component_type}::{language}::{source_norm}"

                for ordinal, occ in item_occ.iterrows():
                    record = record_lookup.loc[occ["record_key"]]
                    lid_rng = random.Random(stable_seed(group_id, record["record_key"], base_seed=seed))
                    lid = lid_rng.choice(list(record["shared_lids"]))
                    base_row = record_to_base_row(record, lid)
                    is_corrupt = ordinal in corrupted_indices
                    corruption_type = (
                        f"qa1_{component_type}_inconsistency"
                        if is_corrupt
                        else "qa1_consistency_clean_control"
                    )
                    variant = base_variant(base_row, language, corruption_type, ordinal)
                    # The same base record can participate in several entity/predicate
                    # consistency groups. Include the group identity in the ordinal salt
                    # so that clean controls from different groups cannot collide.
                    consistency_ordinal = stable_seed(
                        "consistency-variant",
                        group_id,
                        record["record_key"],
                        ordinal,
                        base_seed=seed,
                    )
                    variant["variant_id"] = make_variant_id(
                        base_row["base_pair_key"],
                        language,
                        corruption_type,
                        consistency_ordinal,
                    )
                    variant["consistency_group_id"] = group_id
                    variant["generation_notes"] = (
                        f"Dedicated corpus-level {component_type} consistency group; "
                        f"{n_corrupt}/{len(item_occ)} occurrences perturbed."
                    )

                    if is_corrupt:
                        updated_triples = list(variant["clean_target_triples"])
                        source_triples = variant["source_triples"]
                        changed_pairs = []
                        for ti, source_triple in enumerate(source_triples):
                            source_parts = split_triple(source_triple)
                            target_parts = list(split_triple(updated_triples[ti]))
                            roles = [1] if component_type == "predicate" else [0, 2]
                            for role in roles:
                                if normalize_label(source_parts[role]) == source_norm:
                                    old = target_parts[role]
                                    new = f"{display_label(old)}{LANG_CONFIG[language]['alt_suffix']}"
                                    target_parts[role] = new.replace(" ", "_") if "_" in old else new
                                    changed_pairs.append((old, target_parts[role]))
                            updated_triples[ti] = join_triple(target_parts)
                        if not changed_pairs:
                            continue
                        variant["variant_target_triples"] = updated_triples
                        variant["changed_value_before"] = " || ".join(pair[0] for pair in changed_pairs)
                        variant["changed_value_after"] = " || ".join(pair[1] for pair in changed_pairs)
                    variants.append(variant)
    return variants


def generate_benign_consistency_track(
    records_df: pd.DataFrame,
    items_per_type: int,
    occurrences_per_item: int,
    seed: int,
) -> list[dict[str, Any]]:
    """Create negative-control groups from naturally occurring target variants.

    These groups are not assumed correct automatically; they are exported for
    manual verification. They test whether the consistency metric behaves as a
    review signal rather than an automatic correctness classifier.
    """
    variants: list[dict[str, Any]] = []
    if items_per_type <= 0:
        return variants
    record_lookup = records_df.set_index("record_key", drop=False)

    for component_type in ("predicate", "entity"):
        occurrences = component_occurrence_table(records_df, component_type)
        if occurrences.empty:
            continue
        occurrences["target_norm"] = occurrences["target_value"].map(normalize_label)
        for language in TARGET_LANGS:
            lang_occ = occurrences[occurrences["language"] == language].copy()
            variation = (
                lang_occ.groupby("source_norm")
                .agg(
                    record_count=("record_key", "nunique"),
                    target_form_count=("target_norm", "nunique"),
                )
            )
            eligible = variation[
                (variation["record_count"] >= 2)
                & (variation["target_form_count"] >= 2)
            ].sort_values(["target_form_count", "record_count"], ascending=False)
            for source_norm in eligible.head(items_per_type).index:
                raw_occ = lang_occ[
                    lang_occ["source_norm"] == source_norm
                ].drop_duplicates(["record_key", "target_norm"])
                # Preserve at least one naturally occurring example of every
                # target form, then fill the remaining group slots at random.
                mandatory_parts = []
                for target_norm, form_group in raw_occ.groupby("target_norm"):
                    mandatory_parts.append(
                        form_group.sample(
                            n=1,
                            random_state=stable_seed(
                                "benign-form",
                                component_type,
                                language,
                                source_norm,
                                target_norm,
                                base_seed=seed,
                            ),
                        )
                    )
                mandatory = pd.concat(mandatory_parts, ignore_index=True)
                remaining_pool = raw_occ[
                    ~raw_occ["record_key"].isin(mandatory["record_key"])
                ].drop_duplicates("record_key")
                remaining_n = max(
                    0,
                    min(occurrences_per_item, raw_occ["record_key"].nunique())
                    - len(mandatory),
                )
                if remaining_n and not remaining_pool.empty:
                    extra = remaining_pool.sample(
                        n=min(remaining_n, len(remaining_pool)),
                        random_state=stable_seed(
                            "benign-fill",
                            component_type,
                            language,
                            source_norm,
                            base_seed=seed,
                        ),
                    )
                    item_occ = pd.concat([mandatory, extra], ignore_index=True)
                else:
                    item_occ = mandatory.reset_index(drop=True)
                group_id = f"benign::{component_type}::{language}::{source_norm}"
                corruption_type = f"qa1_{component_type}_benign_variation"
                for ordinal, occ in item_occ.iterrows():
                    record = record_lookup.loc[occ["record_key"]]
                    lid = random.Random(
                        stable_seed(group_id, record["record_key"], base_seed=seed)
                    ).choice(list(record["shared_lids"]))
                    base_row = record_to_base_row(record, lid)
                    variant = base_variant(base_row, language, corruption_type, ordinal)
                    variant["variant_id"] = make_variant_id(
                        base_row["base_pair_key"],
                        language,
                        corruption_type,
                        stable_seed(group_id, record["record_key"], ordinal, base_seed=seed),
                    )
                    variant["consistency_group_id"] = group_id
                    variant["generation_notes"] = (
                        "Naturally occurring source-to-target mapping variation; "
                        "manual verification required before treating as benign."
                    )
                    variants.append(variant)
    return variants


def add_clean_controls(base_df: pd.DataFrame) -> list[dict[str, Any]]:
    rows = []
    for language in TARGET_LANGS:
        for ordinal, row in enumerate(base_df.to_dict("records")):
            rows.append(base_variant(row, language, "clean", ordinal))
    return rows


In [ ]:


def validate_manifest(manifest: pd.DataFrame) -> pd.DataFrame:
    issues: list[dict[str, Any]] = []
    if manifest["variant_id"].duplicated().any():
        for variant_id in manifest.loc[manifest["variant_id"].duplicated(False), "variant_id"]:
            issues.append({"variant_id": variant_id, "issue": "duplicate variant_id"})

    for row in manifest.itertuples(index=False):
        if len(row.variant_lids) != len(row.variant_target_texts):
            issues.append({
                "variant_id": row.variant_id,
                "issue": (
                    "target LID and target-text collections have "
                    "different lengths"
                ),
            })
        if row.corruption_type == "qa1_missing_text":
            if all(normalize_space(text) for text in row.variant_target_texts):
                issues.append({
                    "variant_id": row.variant_id,
                    "issue": "missing-text corruption did not empty a target text",
                })
        if row.corruption_type == "qa1_wrong_lid":
            if row.source_lids == row.variant_lids:
                issues.append({"variant_id": row.variant_id, "issue": "wrong-LID corruption did not change the LID collection"})
        if row.corruption_type == "qa1_duplicate_lid":
            if len(row.variant_lids) == len(set(row.variant_lids)):
                issues.append({"variant_id": row.variant_id, "issue": "duplicate-LID corruption did not create an actual duplicate"})
        if row.corruption_type == "clean":
            if row.clean_target_triples != row.variant_target_triples or row.clean_target_text != row.variant_target_text:
                issues.append({"variant_id": row.variant_id, "issue": "clean control changed"})
        if row.is_corrupted and row.corruption_type not in {"qa1_wrong_lid", "qa1_duplicate_lid"}:
            changed = (
                row.clean_target_triples != row.variant_target_triples
                or row.clean_target_text != row.variant_target_text
                or row.source_lid != row.variant_lid
                or bool(row.xml_duplicate_target_lex)
            )
            if not changed:
                issues.append({"variant_id": row.variant_id, "issue": "corruption produced no change"})

        if row.is_benign:
            if row.corruption_type == "benign_alternative_reference":
                if (
                    row.clean_target_triples != row.variant_target_triples
                    or row.source_lid != row.variant_lid
                    or bool(row.xml_duplicate_target_lex)
                ):
                    issues.append({
                        "variant_id": row.variant_id,
                        "issue": "benign alternative reference changed triples or alignment metadata",
                    })
                if normalize_space(row.clean_target_text) == normalize_space(row.variant_target_text):
                    issues.append({
                        "variant_id": row.variant_id,
                        "issue": "benign alternative reference produced no textual variation",
                    })
            elif row.corruption_type in {
                "qa1_predicate_benign_variation",
                "qa1_entity_benign_variation",
            }:
                if (
                    row.clean_target_triples != row.variant_target_triples
                    or row.clean_target_text != row.variant_target_text
                    or row.source_lid != row.variant_lid
                ):
                    issues.append({
                        "variant_id": row.variant_id,
                        "issue": "benign consistency control was modified",
                    })
        if len(row.source_triples) != int(row.expected_triple_count):
            issues.append({"variant_id": row.variant_id, "issue": "source modified-triple count differs from declared size"})
        if row.language not in TARGET_LANGS:
            issues.append({"variant_id": row.variant_id, "issue": "unsupported language"})
    return pd.DataFrame(issues)


def json_serializable_record(row: Mapping[str, Any]) -> dict[str, Any]:
    result = {}
    for key, value in row.items():
        if isinstance(value, (np.integer,)):
            value = int(value)
        elif isinstance(value, (np.floating,)):
            value = float(value)
        elif isinstance(value, np.ndarray):
            value = value.tolist()
        result[key] = value
    return result


def write_manifest(manifest: pd.DataFrame, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest.to_pickle(output_dir / "noisy_manifest.pkl")
    try:
        manifest.to_parquet(output_dir / "noisy_manifest.parquet", index=False)
    except Exception as exc:
        (output_dir / "PARQUET_NOT_WRITTEN.txt").write_text(str(exc), encoding="utf-8")

    csv_df = manifest.copy()
    list_columns = [
        "expected_metrics",
        "source_triples",
        "clean_target_triples",
        "variant_target_triples",
        "other_language_triples",
        "enbt_triples",
        "source_lids",
        "variant_lids",
        "variant_target_texts",
    ]
    for column in list_columns:
        csv_df[column] = csv_df[column].map(lambda value: json.dumps(value, ensure_ascii=False))
    csv_df.to_csv(output_dir / "noisy_manifest.csv", index=False)

    with (output_dir / "noisy_manifest.jsonl").open("w", encoding="utf-8") as handle:
        for record in manifest.to_dict("records"):
            handle.write(json.dumps(json_serializable_record(record), ensure_ascii=False) + "\n")


def add_triple_set(entry: ET.Element, set_tag: str, triple_tag: str, triples: Sequence[str]) -> None:
    triple_set = ET.SubElement(entry, set_tag)
    for triple in triples:
        node = ET.SubElement(triple_set, triple_tag)
        node.text = str(triple)


def write_track_xml(manifest: pd.DataFrame, output_dir: Path) -> None:
    xml_root = output_dir / "xml"
    for (track, split), group in manifest.groupby(["qa_track", "split"], dropna=False):
        track_dir = xml_root / str(track).replace("/", "_").replace(" ", "_") / str(split)
        track_dir.mkdir(parents=True, exist_ok=True)
        root = ET.Element("benchmark")
        entries_node = ET.SubElement(root, "entries")

        for row in group.itertuples(index=False):
            attrs = {
                "category": str(row.category),
                "eid": str(row.variant_id),
                "source_eid": str(row.source_eid),
                "source_record_key": str(row.record_key),
                "noise_track": str(row.qa_track),
                "corruption_type": str(row.corruption_type),
                "severity": str(row.severity),
                "size": str(row.expected_triple_count),
            }
            if row.shape:
                attrs["shape"] = str(row.shape)
            if row.shape_type:
                attrs["shape_type"] = str(row.shape_type)
            entry = ET.SubElement(entries_node, "entry", attrs)

            # English RDF source is exclusively the canonical modified triple set.
            add_triple_set(entry, "modifiedtripleset", "mtriple", row.source_triples)

            if row.language == "es":
                es_triples = row.variant_target_triples
                ca_triples = row.other_language_triples
                es_text = row.variant_target_text
                ca_text = row.other_language_text
                es_lid = row.variant_lid
                ca_lid = row.source_lid
            else:
                es_triples = row.other_language_triples
                ca_triples = row.variant_target_triples
                es_text = row.other_language_text
                ca_text = row.variant_target_text
                es_lid = row.source_lid
                ca_lid = row.variant_lid

            add_triple_set(entry, "spanishtripleset", "striple", es_triples)
            add_triple_set(entry, "catalantripleset", "ctriple", ca_triples)
            add_triple_set(entry, "enbttripleset", "bttriple", row.enbt_triples)

            en_lex = ET.SubElement(entry, "lex", {"lang": "en", "lid": str(row.source_lid)})
            en_lex.text = row.source_text
            es_lex = ET.SubElement(entry, "lex", {"lang": "es", "lid": str(es_lid)})
            es_lex.text = es_text
            ca_lex = ET.SubElement(entry, "lex", {"lang": "ca", "lid": str(ca_lid)})
            ca_lex.text = ca_text
            enbt_lex = ET.SubElement(entry, "lex", {"lang": "en_bt", "lid": str(row.source_lid)})
            enbt_lex.text = row.enbt_text

            if row.xml_duplicate_target_lex:
                duplicate_lang = row.language
                duplicate_lid = row.variant_lid
                duplicate_text = row.variant_target_text
                duplicate = ET.SubElement(entry, "lex", {"lang": duplicate_lang, "lid": str(duplicate_lid)})
                duplicate.text = duplicate_text

        tree = ET.ElementTree(root)
        ET.indent(tree, space="  ")
        tree.write(track_dir / f"{track.lower().replace('-', '_')}_{split}.xml", encoding="utf-8", xml_declaration=True)


In [ ]:



def write_evaluation_views(manifest: pd.DataFrame, output_dir: Path) -> None:
    """Write convenient XML views containing clean controls and the relevant track."""
    views = {
        "QA1": {"CLEAN", "QA1-STRUCT", "QA1-CONSISTENCY"},
        "QA2": {"CLEAN", "QA2-SEM", "BENIGN-VAR"},
        "QA3": {"CLEAN", "QA3-LANG", "BENIGN-VAR"},
    }
    views_root = output_dir / "xml_views"
    for view_name, tracks in views.items():
        view_df = manifest[manifest["qa_track"].isin(tracks)]
        for split, group in view_df.groupby("split", dropna=False):
            split_dir = views_root / view_name / str(split)
            split_dir.mkdir(parents=True, exist_ok=True)
            root = ET.Element("benchmark")
            entries_node = ET.SubElement(root, "entries")
            for row in group.itertuples(index=False):
                attrs = {
                    "category": str(row.category),
                    "eid": str(row.variant_id),
                    "source_eid": str(row.source_eid),
                    "source_record_key": str(row.record_key),
                    "noise_track": str(row.qa_track),
                    "corruption_type": str(row.corruption_type),
                    "severity": str(row.severity),
                    "size": str(row.expected_triple_count),
                }
                if row.shape:
                    attrs["shape"] = str(row.shape)
                if row.shape_type:
                    attrs["shape_type"] = str(row.shape_type)
                entry = ET.SubElement(entries_node, "entry", attrs)
                add_triple_set(entry, "modifiedtripleset", "mtriple", row.source_triples)

                if row.language == "es":
                    es_triples, ca_triples = row.variant_target_triples, row.other_language_triples
                    es_text, ca_text = row.variant_target_text, row.other_language_text
                    es_lid, ca_lid = row.variant_lid, row.source_lid
                else:
                    es_triples, ca_triples = row.other_language_triples, row.variant_target_triples
                    es_text, ca_text = row.other_language_text, row.variant_target_text
                    es_lid, ca_lid = row.source_lid, row.variant_lid

                add_triple_set(entry, "spanishtripleset", "striple", es_triples)
                add_triple_set(entry, "catalantripleset", "ctriple", ca_triples)
                add_triple_set(entry, "enbttripleset", "bttriple", row.enbt_triples)

                for lang, lid, text in [
                    ("en", row.source_lid, row.source_text),
                    ("es", es_lid, es_text),
                    ("ca", ca_lid, ca_text),
                    ("en_bt", row.source_lid, row.enbt_text),
                ]:
                    lex = ET.SubElement(entry, "lex", {"lang": lang, "lid": str(lid)})
                    lex.text = text
                if row.xml_duplicate_target_lex:
                    duplicate = ET.SubElement(
                        entry, "lex", {"lang": row.language, "lid": str(row.variant_lid)}
                    )
                    duplicate.text = row.variant_target_text

            tree = ET.ElementTree(root)
            ET.indent(tree, space="  ")
            tree.write(
                split_dir / f"auditstress_{view_name.lower()}_{split}.xml",
                encoding="utf-8",
                xml_declaration=True,
            )


def write_language_specific_views(manifest: pd.DataFrame, output_dir: Path) -> None:
    """Write QA views filtered to the language intentionally modified in each row."""
    views = {
        "QA1": {"CLEAN", "QA1-STRUCT", "QA1-CONSISTENCY"},
        "QA2": {"CLEAN", "QA2-SEM", "BENIGN-VAR"},
        "QA3": {"CLEAN", "QA3-LANG", "BENIGN-VAR"},
    }
    root_dir = output_dir / "xml_views_by_language"
    for view_name, tracks in views.items():
        for language in TARGET_LANGS:
            view_df = manifest[
                manifest["qa_track"].isin(tracks) & manifest["language"].eq(language)
            ]
            for split, group in view_df.groupby("split", dropna=False):
                split_dir = root_dir / view_name / language / str(split)
                split_dir.mkdir(parents=True, exist_ok=True)
                root = ET.Element("benchmark")
                entries_node = ET.SubElement(root, "entries")
                for row in group.itertuples(index=False):
                    attrs = {
                        "category": str(row.category),
                        "eid": str(row.variant_id),
                        "source_eid": str(row.source_eid),
                        "source_record_key": str(row.record_key),
                        "noise_track": str(row.qa_track),
                        "corruption_type": str(row.corruption_type),
                        "intended_language": str(row.language),
                        "severity": str(row.severity),
                        "size": str(row.expected_triple_count),
                    }
                    if row.shape:
                        attrs["shape"] = str(row.shape)
                    if row.shape_type:
                        attrs["shape_type"] = str(row.shape_type)
                    entry = ET.SubElement(entries_node, "entry", attrs)
                    add_triple_set(entry, "modifiedtripleset", "mtriple", row.source_triples)
                    if row.language == "es":
                        es_triples, ca_triples = row.variant_target_triples, row.other_language_triples
                        es_text, ca_text = row.variant_target_text, row.other_language_text
                        es_lid, ca_lid = row.variant_lid, row.source_lid
                    else:
                        es_triples, ca_triples = row.other_language_triples, row.variant_target_triples
                        es_text, ca_text = row.other_language_text, row.variant_target_text
                        es_lid, ca_lid = row.source_lid, row.variant_lid
                    add_triple_set(entry, "spanishtripleset", "striple", es_triples)
                    add_triple_set(entry, "catalantripleset", "ctriple", ca_triples)
                    add_triple_set(entry, "enbttripleset", "bttriple", row.enbt_triples)
                    for lang, lid, text in [
                        ("en", row.source_lid, row.source_text),
                        ("es", es_lid, es_text),
                        ("ca", ca_lid, ca_text),
                        ("en_bt", row.source_lid, row.enbt_text),
                    ]:
                        lex = ET.SubElement(entry, "lex", {"lang": lang, "lid": str(lid)})
                        lex.text = text
                    if row.xml_duplicate_target_lex:
                        duplicate = ET.SubElement(
                            entry, "lex", {"lang": row.language, "lid": str(row.variant_lid)}
                        )
                        duplicate.text = row.variant_target_text
                tree = ET.ElementTree(root)
                ET.indent(tree, space="  ")
                tree.write(
                    split_dir / f"auditstress_{view_name.lower()}_{language}_{split}.xml",
                    encoding="utf-8",
                    xml_declaration=True,
                )

def build_summary(manifest: pd.DataFrame) -> pd.DataFrame:
    return (
        manifest.groupby(
            ["qa_track", "qa_subqa", "corruption_type", "severity", "language"],
            dropna=False,
        )
        .agg(
            variants=("variant_id", "size"),
            base_pairs=("base_pair_key", "nunique"),
            records=("record_key", "nunique"),
        )
        .reset_index()
        .sort_values(["qa_track", "qa_subqa", "corruption_type", "language"])
    )


def select_manual_verification_sample(manifest: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    candidates = manifest[manifest["corruption_type"] != "clean"].copy()
    if candidates.empty:
        return candidates
    strata = ["corruption_type", "language"]
    groups = list(candidates.groupby(strata))
    per_group = max(1, math.floor(n / max(1, len(groups))))
    pieces = []
    for key, group in groups:
        pieces.append(group.sample(n=min(per_group, len(group)), random_state=stable_seed(key, base_seed=seed)))
    sample = pd.concat(pieces, ignore_index=True)
    if len(sample) < n:
        remaining = candidates[~candidates["variant_id"].isin(sample["variant_id"])]
        if not remaining.empty:
            sample = pd.concat(
                [sample, remaining.sample(n=min(n - len(sample), len(remaining)), random_state=seed + 17)],
                ignore_index=True,
            )
    sample = sample.sample(frac=1, random_state=seed).head(n).reset_index(drop=True)
    sample["manual_corruption_present"] = ""
    sample["manual_only_intended_corruption"] = ""
    sample["manual_severity_correct"] = ""
    sample["manual_benign_variation_acceptable"] = ""
    sample["manual_notes"] = ""
    return sample


def write_readme(config: BuildConfig, summary: pd.DataFrame, output_dir: Path) -> None:
    total = int(summary["variants"].sum())
    readme = f"""# WebNLG AuditStress controlled perturbation benchmark

This benchmark was generated automatically from the repository's aligned English,
Spanish, and Catalan WebNLG data. English RDF is represented exclusively through
`modifiedtripleset/mtriple`.

## Build configuration

- Base aligned records: {config.base_size}
- Random seed: {config.seed}
- Total generated language-specific variants: {total}
- QA1 consistency items per component type and language: {config.consistency_items_per_type}
- QA1 consistency occurrences per item: {config.consistency_occurrences_per_item}
- QA1 consistency corruption fraction: {config.consistency_corruption_fraction:.2f}
- Benign consistency items per component type: {config.benign_consistency_items_per_type}
- Excluded records from prior benchmark: {len(load_excluded_record_keys(config.exclude_manifest))}

## Files

- `noisy_manifest.pkl`: lossless pandas representation.
- `noisy_manifest.parquet`: portable columnar representation when `pyarrow` or
  `fastparquet` is available.
- `noisy_manifest.csv`: list-valued fields encoded as JSON strings.
- `noisy_manifest.jsonl`: one JSON object per variant.
- `generation_summary.csv`: counts by track, corruption, severity, and language.
- `manual_verification_sample.csv`: stratified sample for verifying that automatic
  perturbations contain only the intended defect.
- `xml/`: WebNLG-style XML grouped by QA track and split.
- `xml_views/QA1`, `xml_views/QA2`, and `xml_views/QA3`: combined convenience views.
- `xml_views_by_language/<QA>/<es|ca>`: recommended views containing only
  variants whose intended modified language is Spanish or Catalan.

## Intended use

The challenge tracks validate the sensitivity and selectivity of the audit
indicators. They are not alternative training datasets and should not be used to
repeat the downstream generation experiments. Thresholds and metric
implementations should be frozen before evaluating these perturbations.

Run each QA notebook only on its corresponding XML track, or use the manifest to
compute paired clean-versus-corrupted changes. The `BENIGN-VAR` track contains
naturally occurring alternative references for the same RDF entry and serves as a
negative control against over-penalising legitimate linguistic variation.
"""
    (output_dir / "README.md").write_text(readme, encoding="utf-8")


In [ ]:


def build_benchmark(config: BuildConfig) -> dict[str, Any]:
    repo_root = Path(config.repo_root).expanduser().resolve()
    output_dir = Path(config.output_dir).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    records_df, ignored_xml, parse_errors = parse_repository(repo_root)
    excluded_record_keys = load_excluded_record_keys(config.exclude_manifest)
    if excluded_record_keys:
        records_df = records_df[~records_df["record_key"].isin(excluded_record_keys)].reset_index(drop=True)
        if records_df.empty:
            raise NoisyBenchmarkError("All repository records were excluded by --exclude-manifest.")
    base_candidates = one_lexicalisation_per_record(records_df, config.seed)
    base_df = proportional_stratified_sample(base_candidates, config.base_size, config.seed)

    counts = dict(DEFAULT_COUNTS)
    if config.counts_per_language:
        counts.update({str(key): int(value) for key, value in config.counts_per_language.items()})

    standard_variants = generate_standard_variants(base_df, counts, config.seed)
    standard_variants.extend(generate_nested_omission_variants(base_df, counts, config.seed))
    standard_variants.extend(generate_paired_addition_variants(base_df, counts, config.seed))
    consistency_variants = generate_consistency_track(
        records_df,
        items_per_type=config.consistency_items_per_type,
        occurrences_per_item=config.consistency_occurrences_per_item,
        corruption_fraction=config.consistency_corruption_fraction,
        seed=config.seed,
    )
    benign_consistency_variants = generate_benign_consistency_track(
        records_df,
        items_per_type=config.benign_consistency_items_per_type,
        occurrences_per_item=config.consistency_occurrences_per_item,
        seed=config.seed,
    )
    clean_controls = add_clean_controls(base_df)

    manifest = pd.DataFrame(
        clean_controls + standard_variants + consistency_variants + benign_consistency_variants
    )
    manifest = manifest.sort_values(["qa_track", "corruption_type", "language", "variant_id"]).reset_index(drop=True)
    issues = validate_manifest(manifest)
    if not issues.empty:
        issues.to_csv(output_dir / "manifest_validation_issues.csv", index=False)
        raise NoisyBenchmarkError(
            f"Manifest validation failed with {len(issues)} issue(s). See manifest_validation_issues.csv."
        )

    write_manifest(manifest, output_dir)
    base_df.to_pickle(output_dir / "clean_base.pkl")
    base_csv = base_df.copy()
    for column in ["source_triples", "es_triples", "ca_triples", "enbt_triples"]:
        base_csv[column] = base_csv[column].map(lambda value: json.dumps(value, ensure_ascii=False))
    for column in ["all_lex_en", "all_lex_es", "all_lex_ca"]:
        base_csv[column] = base_csv[column].map(lambda value: json.dumps(value, ensure_ascii=False))
    base_csv.to_csv(output_dir / "clean_base.csv", index=False)
    base_df[[
        "base_pair_key",
        "record_key",
        "split",
        "declared_size_raw",
        "declared_size",
        "declared_size_valid",
        "expected_triple_count",
        "category",
    ]].to_csv(
        output_dir / "selected_base_records.csv", index=False
    )

    ignored_xml.to_csv(output_dir / "ignored_xml_files.csv", index=False)
    parse_errors.to_csv(output_dir / "xml_parse_errors.csv", index=False)

    summary = build_summary(manifest)
    summary.to_csv(output_dir / "generation_summary.csv", index=False)

    catalog = pd.DataFrame([
        {"corruption_type": key, **value}
        for key, value in TRACK_EXPECTATIONS.items()
    ])
    catalog["expected_metrics"] = catalog["expected_metrics"].map(
        lambda value: json.dumps(value, ensure_ascii=False)
    )
    catalog.to_csv(output_dir / "corruption_catalog.csv", index=False)

    manual_sample = select_manual_verification_sample(
        manifest, config.manual_verification_sample, config.seed
    )
    manual_csv = manual_sample.copy()
    for column in [
        "expected_metrics",
        "source_triples",
        "clean_target_triples",
        "variant_target_triples",
        "other_language_triples",
        "enbt_triples",
        "source_lids",
        "variant_lids",
        "variant_target_texts",
    ]:
        manual_csv[column] = manual_csv[column].map(lambda value: json.dumps(value, ensure_ascii=False))
    manual_csv.to_csv(output_dir / "manual_verification_sample.csv", index=False)

    if config.write_xml:
        write_track_xml(manifest, output_dir)
        write_evaluation_views(manifest, output_dir)
        write_language_specific_views(manifest, output_dir)

    config_payload = asdict(config)
    config_payload["counts_per_language"] = counts
    (output_dir / "build_config.json").write_text(
        json.dumps(config_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    write_readme(config, summary, output_dir)

    return {
        "records": records_df,
        "base": base_df,
        "manifest": manifest,
        "summary": summary,
        "manual_sample": manual_sample,
        "ignored_xml": ignored_xml,
        "parse_errors": parse_errors,
        "output_dir": output_dir,
    }


def parse_counts(value: str | None) -> dict[str, int] | None:
    if not value:
        return None
    stripped = value.lstrip()
    if stripped.startswith("{"):
        payload = json.loads(value)
    else:
        path = Path(value)
        payload = json.loads(path.read_text(encoding="utf-8"))
    return {str(key): int(item) for key, item in payload.items()}


def main() -> None:
    parser = argparse.ArgumentParser(description="Create the WebNLG AuditStress controlled noisy benchmark.")
    parser.add_argument("--repo-root", required=True, help="Repository root containing WebNLG_CA_BT.")
    parser.add_argument("--output-dir", required=True, help="Output directory.")
    parser.add_argument("--base-size", type=int, default=1000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--counts",
        default=None,
        help="JSON object or path to a JSON file overriding per-language corruption counts.",
    )
    parser.add_argument("--consistency-items", type=int, default=10)
    parser.add_argument("--consistency-occurrences", type=int, default=30)
    parser.add_argument("--consistency-fraction", type=float, default=0.30)
    parser.add_argument("--benign-consistency-items", type=int, default=5)
    parser.add_argument(
        "--exclude-manifest",
        default=None,
        help="Prior AuditStress manifest or benchmark directory whose record_key values must be excluded.",
    )
    parser.add_argument("--manual-sample", type=int, default=300)
    parser.add_argument("--no-xml", action="store_true")
    args = parser.parse_args()

    config = BuildConfig(
        repo_root=args.repo_root,
        output_dir=args.output_dir,
        base_size=args.base_size,
        seed=args.seed,
        counts_per_language=parse_counts(args.counts),
        consistency_items_per_type=args.consistency_items,
        consistency_occurrences_per_item=args.consistency_occurrences,
        consistency_corruption_fraction=args.consistency_fraction,
        benign_consistency_items_per_type=args.benign_consistency_items,
        manual_verification_sample=args.manual_sample,
        exclude_manifest=args.exclude_manifest,
        write_xml=not args.no_xml,
    )
    result = build_benchmark(config)
    print(result["summary"].to_string(index=False))
    print(f"\nWrote benchmark to: {result['output_dir']}")


## 1.1 Generate and validate the benchmark

In [ ]:

if RUN_GENERATION:
    config = BuildConfig(
        repo_root=str(REPO_ROOT),
        output_dir=str(BENCHMARK_DIR),
        base_size=BASE_SIZE,
        seed=GENERATION_SEED,
        counts_per_language=COUNTS_PER_LANGUAGE,
        consistency_items_per_type=CONSISTENCY_ITEMS,
        consistency_occurrences_per_item=CONSISTENCY_OCCURRENCES,
        consistency_corruption_fraction=CONSISTENCY_CORRUPTION_FRACTION,
        benign_consistency_items_per_type=BENIGN_CONSISTENCY_ITEMS,
        manual_verification_sample=MANUAL_SAMPLE,
        exclude_manifest=(str(EXCLUDE_MANIFEST) if EXCLUDE_MANIFEST else None),
        write_xml=WRITE_XML,
    )
    generation_result = build_benchmark(config)
    display(generation_result['summary'])
    print('Benchmark written to:', generation_result['output_dir'])
else:
    print('Generation skipped. Using existing benchmark:', BENCHMARK_DIR)

manifest_path = BENCHMARK_DIR / 'noisy_manifest.pkl'
if manifest_path.exists():
    generated_manifest = pd.read_pickle(manifest_path)
    corruption_counts = generated_manifest['corruption_type'].value_counts()
    assert int(corruption_counts.get('qa2_unsupported_addition', 0)) == 0
    assert int(corruption_counts.get('qa2_unsupported_addition_plausible', 0)) > 0
    assert int(corruption_counts.get('qa2_unsupported_addition_unrelated', 0)) > 0

    if EXCLUDE_MANIFEST is not None:
        prior_path = EXCLUDE_MANIFEST
        if prior_path.is_dir():
            prior_path = prior_path / 'noisy_manifest.pkl'
        prior_manifest = pd.read_pickle(prior_path)
        overlap = set(prior_manifest['record_key']) & set(generated_manifest['record_key'])
        print('Excluded-record overlap:', len(overlap))
        assert not overlap

    display(
        generated_manifest.groupby(
            ['qa_track', 'language_label', 'corruption_type'],
            as_index=False,
        ).size().rename(columns={'size': 'variants'})
    )
else:
    print('No benchmark manifest is available yet.')



# 2. Stress-audit implementation

The following cells contain the complete aligned runner. Its shared metric
functions are overridden by `audit_metric_core.py`, which is also imported by
the three original-resource notebooks.


In [ ]:
#!/usr/bin/env python3
"""Run QA1–QA3 audit validation on a WebNLG AuditStress manifest.

The benchmark validates the audit framework, not downstream model robustness.
English RDF is always taken from `modifiedtripleset`, represented in the
manifest by `source_triples`.

Recommended final configuration:
    --semantic-backend sentence_transformer
    --semantic-model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

The TF–IDF backend is for execution tests only.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
import argparse
import ast
import hashlib
import html
import json
import math
import os
import re
import unicodedata
import warnings
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import joblib
except ImportError:
    joblib = None

GENERATOR_COMPATIBILITY = "1.3"
METRIC_CORE_COMPATIBILITY = "audit-text-aligned-2.7"
TARGET_LANGS = ("es", "ca")
LANG_LABELS = {"es": "Spanish", "ca": "Catalan", "en": "English"}
RANDOM_SEED = 42


class AuditStressError(RuntimeError):
    pass


def stable_hash_int(*parts: object) -> int:
    digest = hashlib.sha1("||".join(map(str, parts)).encode("utf-8")).hexdigest()[:8]
    return int(digest, 16)


def normalize_space(value: object) -> str:
    if value is None:
        return ""
    try:
        if isinstance(value, float) and np.isnan(value):
            return ""
    except (TypeError, ValueError):
        pass
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_label(value: object) -> str:
    value = html.unescape(str(value or "")).strip()
    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"@[A-Za-z-]+\s*$", "", value)
    value = value.replace("_", " ")
    value = re.sub(r"\s+", " ", value)
    return value.strip(" \t\n\r\"'").casefold()


def split_triple(value: object) -> tuple[str, str, str]:
    parts = [part.strip() for part in str(value or "").split("|")]
    if len(parts) < 3:
        return "", "", ""
    return parts[0], parts[1], "|".join(parts[2:]).strip()


def clean_rdf_label(value: object) -> str:
    value = html.unescape(str(value or "")).strip()
    value = re.sub(r"\^\^xsd:[A-Za-z]+\s*$", "", value)
    value = re.sub(r"@[A-Za-z-]+\s*$", "", value)
    value = value.strip("\"' ")
    value = value.replace("_", " ")
    return re.sub(r"\s+", " ", value).strip()


def serialize_triple(triple: str) -> str:
    subject, predicate, obj = map(clean_rdf_label, split_triple(triple))
    return f"[S] {subject} [P] {predicate} [O] {obj}"


def serialize_triple_set(triples: list[str]) -> str:
    return " ".join(serialize_triple(triple) for triple in triples)


def ensure_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return []
        try:
            parsed = json.loads(stripped)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        try:
            parsed = ast.literal_eval(stripped)
            if isinstance(parsed, (list, tuple)):
                return list(parsed)
        except Exception:
            pass
    raise AuditStressError(f"Cannot decode list-valued field: {value!r}")



def load_manifest(benchmark_dir: Path) -> pd.DataFrame:
    pkl = benchmark_dir / "noisy_manifest.pkl"
    csv = benchmark_dir / "noisy_manifest.csv"
    if pkl.exists():
        frame = pd.read_pickle(pkl)
    elif csv.exists():
        frame = pd.read_csv(csv)
        list_columns = [
            "expected_metrics",
            "source_triples",
            "clean_target_triples",
            "variant_target_triples",
            "other_language_triples",
            "enbt_triples",
            "source_lids",
            "variant_lids",
            "variant_target_texts",
        ]
        for column in list_columns:
            if column in frame.columns:
                frame[column] = frame[column].map(ensure_list)
    else:
        raise FileNotFoundError(
            f"Neither noisy_manifest.pkl nor noisy_manifest.csv exists under {benchmark_dir}"
        )

    required = {
        "variant_id", "base_pair_key", "record_key", "language", "qa_track",
        "corruption_type", "severity", "declared_size", "source_lid",
        "variant_lid", "source_triples", "clean_target_triples",
        "variant_target_triples", "source_text", "clean_target_text",
        "variant_target_text", "xml_duplicate_target_lex",
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise AuditStressError(f"Manifest is missing required columns: {missing}")

    for column in [
        "source_triples",
        "clean_target_triples",
        "variant_target_triples",
    ]:
        frame[column] = frame[column].map(ensure_list)

    # New QA1.1 entry-level collections. Old manifests remain loadable and
    # are upgraded deterministically from their scalar fields.
    if "source_lids" not in frame.columns:
        frame["source_lids"] = frame["source_lid"].map(
            lambda value: [value]
        )
    else:
        frame["source_lids"] = frame["source_lids"].map(ensure_list)

    if "variant_lids" not in frame.columns:
        frame["variant_lids"] = frame["variant_lid"].map(
            lambda value: [value]
        )
    else:
        frame["variant_lids"] = frame["variant_lids"].map(ensure_list)

    if "variant_target_texts" not in frame.columns:
        frame["variant_target_texts"] = frame[
            "variant_target_text"
        ].map(lambda value: [value])
    else:
        frame["variant_target_texts"] = frame[
            "variant_target_texts"
        ].map(ensure_list)

    if "expected_triple_count" not in frame.columns:
        frame["expected_triple_count"] = frame[
            "declared_size"
        ].astype(int)
    if "declared_size_valid" not in frame.columns:
        frame["declared_size_valid"] = True
    if "declared_size_raw" not in frame.columns:
        frame["declared_size_raw"] = frame[
            "declared_size"
        ].astype("string")

    optional_defaults = {
        "is_benign": False,
        "consistency_group_id": "",
        "perturbation_pair_id": "",
        "removed_token_fraction": np.nan,
        "removed_sentence_count": np.nan,
        "original_sentence_count": np.nan,
        "remaining_sentence_count": np.nan,
        "alternative_reference_lid": "",
        "alternative_source_text": "",
    }
    for column, default in optional_defaults.items():
        if column not in frame.columns:
            frame[column] = default

    frame["language_label"] = frame["language"].map(LANG_LABELS)
    return frame.reset_index(drop=True)


def save_table(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def save_figure(fig, output_dir: Path, stem: str) -> None:
    fig.savefig(output_dir / f"{stem}.png", dpi=220, bbox_inches="tight")
    fig.savefig(output_dir / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
# QA1
# ---------------------------------------------------------------------------

QA1_EXPECTED_CHECK = {
    "qa1_delete_triple": "target_triple_parity",
    "qa1_duplicate_triple": "target_triple_parity",
    "qa1_missing_text": "target_text_present",
    "qa1_wrong_lid": "lexicalisation_id_alignment",
    "qa1_duplicate_lid": "lexicalisation_id_unique",
    "qa1_placeholder_component": "target_rdf_well_formed",
    "qa1_missing_component": "target_rdf_well_formed",
    "qa1_source_markup": "target_rdf_well_formed",
}


In [ ]:

def run_qa1(manifest: pd.DataFrame, output_dir: Path) -> dict[str, pd.DataFrame]:
    qa1 = manifest[
        manifest["qa_track"].isin(["CLEAN", "QA1-STRUCT", "QA1-CONSISTENCY"])
    ].copy()

    qa1["source_triple_count"] = qa1["source_triples"].map(len)
    qa1["target_triple_count"] = qa1["variant_target_triples"].map(len)
    qa1["source_declared_parity"] = pd.Series(
        pd.NA,
        index=qa1.index,
        dtype="boolean",
    )
    valid_declared = qa1["declared_size_valid"].fillna(False).astype(bool)
    qa1.loc[valid_declared, "source_declared_parity"] = (
        qa1.loc[valid_declared, "source_triple_count"]
        == qa1.loc[valid_declared, "declared_size"].astype(int)
    )

    qa11_metric_rows = qa1.apply(
        lambda row: pd.Series(
            qa11_core.qa11_entry_metrics(
                expected_triple_count=row["expected_triple_count"],
                english_lids=row["source_lids"],
                target_lids=row["variant_lids"],
                target_triples=row["variant_target_triples"],
                target_texts=row["variant_target_texts"],
                triple_well_formed_fn=triple_well_formed,
            )
        ),
        axis=1,
    )
    qa1 = pd.concat(
        [
            qa1.reset_index(drop=True),
            qa11_metric_rows.reset_index(drop=True),
        ],
        axis=1,
    )

    qa1["expected_check"] = qa1["corruption_type"].map(QA1_EXPECTED_CHECK)
    qa1["detected"] = False
    for check in QA1_EXPECTED_CHECK.values():
        mask = qa1["expected_check"].eq(check)
        qa1.loc[mask, "detected"] = ~qa1.loc[mask, check]

    structural = qa1[qa1["qa_track"].isin(["CLEAN", "QA1-STRUCT"])].copy()
    structural_summary = (
        structural.groupby(
            ["language", "language_label", "qa_track", "corruption_type", "severity"],
            dropna=False,
        )
        .agg(
            variants=("variant_id", "size"),
            rdf_entries=("record_key", "nunique"),
            source_sanity_rate=("source_declared_parity", "mean"),
            integrity_pass_rate=("record_integrity", "mean"),
            target_triple_parity_rate=("target_triple_parity", "mean"),
            target_text_present_rate=("target_text_present", "mean"),
            lexicalisation_id_alignment_rate=("lexicalisation_id_alignment", "mean"),
            lexicalisation_id_unique_rate=("lexicalisation_id_unique", "mean"),
            target_rdf_well_formed_rate=("target_rdf_well_formed", "mean"),
            detection_rate=("detected", "mean"),
        )
        .reset_index()
    )


    consistency_rows = []
    consistency = qa1[
        qa1["qa_track"].eq("QA1-CONSISTENCY")
    ].copy()

    for group_id, group in consistency.groupby(
        "consistency_group_id",
        dropna=True,
    ):
        # The group ID records the reported component class r, target
        # language ell, and normalised English source component x.
        parts = str(group_id).split("::")
        if (
            parts
            and parts[0] == "benign"
            and len(parts) >= 4
        ):
            group_kind = "benign_natural_variation"
            component_type, language = parts[1], parts[2]
            source_norm = "::".join(parts[3:])
        elif len(parts) >= 3:
            group_kind = (
                "controlled_harmful_inconsistency"
            )
            component_type, language = parts[0], parts[1]
            source_norm = "::".join(parts[2:])
        else:
            continue

        # Extract the observed target forms y from positionally aligned
        # source and target triple components. Predicate groups inspect
        # predicate position 1; entity groups inspect subject/object
        # positions 0 and 2. Literal-like values were excluded when the
        # stress groups were generated.
        forms = []
        for row in group.itertuples(index=False):
            for source_triple, target_triple in zip(
                row.source_triples,
                row.variant_target_triples,
            ):
                source = split_triple(source_triple)
                target = split_triple(target_triple)
                positions = (
                    [1]
                    if component_type == "predicate"
                    else [0, 2]
                )
                for position in positions:
                    if (
                        normalize_label(source[position])
                        == source_norm
                    ):
                        target_form = normalize_label(
                            target[position]
                        )
                        if target_form:
                            forms.append(target_form)

        # Shared implementation of:
        #   n_{ell,r}(x), D_{ell,r}(x), and M_{ell,r}(x).
        profile = metric_core.qa12_profile_normalized_forms(
            forms,
            minimum_occurrences=(
                QA12_MIN_RECURRING_OCCURRENCES
            ),
        )
        if profile is None:
            # A group below rho is not a recurring-component profile and
            # therefore lies outside the QA1.2 evaluation set.
            continue

        consistency_rows.append({
            "consistency_group_id": group_id,
            "group_kind": group_kind,
            "is_benign_group": (
                group_kind
                == "benign_natural_variation"
            ),
            "component_type": component_type,
            "language": language,
            "language_label": LANG_LABELS.get(
                language,
                language,
            ),
            "source_normalized": source_norm,
            "minimum_occurrences": (
                QA12_MIN_RECURRING_OCCURRENCES
            ),
            "occurrences": profile["occurrences"],
            "target_form_count": profile[
                "target_form_count"
            ],
            "dominant_target_form": profile[
                "dominant_target_form"
            ],
            "dominant_target_count": profile[
                "dominant_target_count"
            ],
            "dominant_mapping_rate": profile[
                "dominant_mapping_rate"
            ],
            "mapping_review_signal": profile[
                "mapping_review_signal"
            ],
            # M_{ell,r}(x) is only a review signal. In the controlled
            # harmful track, the benchmark uses that signal to test
            # sensitivity. Naturally varying groups are never labelled as
            # detected errors solely because M = 1.
            "review_signal_triggered": profile[
                "mapping_review_signal"
            ],
            "inconsistency_detected": (
                bool(profile["mapping_review_signal"])
                if group_kind
                == "controlled_harmful_inconsistency"
                else np.nan
            ),
            "target_forms": json.dumps(
                profile["target_forms"],
                ensure_ascii=False,
                sort_keys=True,
            ),
        })

    consistency_summary = pd.DataFrame(
        consistency_rows
    )

    if not consistency_summary.empty:
        # Shared implementation of the occurrence-weighted D_w summary.
        consistency_selectivity = (
            metric_core.qa12_summarize_mapping_profiles(
                consistency_summary,
                additional_group_columns=[
                    "group_kind",
                    "is_benign_group",
                ],
            )
            .rename(
                columns={
                    "recurring_source_items": "groups",
                    "mapping_review_rate": (
                        "review_signal_rate"
                    ),
                }
            )
        )
    else:
        consistency_selectivity = pd.DataFrame(
            columns=[
                "group_kind",
                "is_benign_group",
                "language",
                "language_label",
                "component_type",
                "groups",
                "mapping_observations",
                "weighted_dominant_mapping_rate",
                "review_signal_rate",
                "items_selected_for_review",
            ]
        )

    # Dataset-level QA1.2 checks complement the synthetic metric contract.
    if not consistency_summary.empty:
        if not (
            consistency_summary["occurrences"]
            >= QA12_MIN_RECURRING_OCCURRENCES
        ).all():
            raise AssertionError(
                "AuditStress retained a QA1.2 group below rho."
            )

        expected_review = (
            consistency_summary["target_form_count"] > 1
        )
        if not (
            consistency_summary["mapping_review_signal"]
            == expected_review
        ).all():
            raise AssertionError(
                "The QA1.2 review flag does not match the observed "
                "number of target forms."
            )

        benign_mask = consistency_summary[
            "group_kind"
        ].eq("benign_natural_variation")
        if consistency_summary.loc[
            benign_mask,
            "inconsistency_detected",
        ].notna().any():
            raise AssertionError(
                "Benign QA1.2 groups must not receive automatic "
                "error labels."
            )

    save_table(qa1, output_dir / "qa1_variant_metrics.csv")
    save_table(structural_summary, output_dir / "qa1_structural_detection_summary.csv")
    save_table(consistency_summary, output_dir / "qa1_consistency_group_results.csv")
    save_table(consistency_selectivity, output_dir / "qa1_consistency_selectivity_summary.csv")

    if not structural_summary.empty:
        plot = structural_summary[
            structural_summary["qa_track"].eq("QA1-STRUCT")
        ].pivot_table(
            index="corruption_type", columns="language_label", values="detection_rate"
        )
        fig, ax = plt.subplots(figsize=(9, 5))
        plot.plot(kind="bar", ax=ax, rot=30)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Detection rate")
        ax.set_xlabel("")
        ax.set_title("QA1 controlled structural-corruption detection")
        ax.grid(axis="y", alpha=0.2)
        fig.tight_layout()
        save_figure(fig, output_dir, "qa1_structural_detection_rate")

    return {
        "variant_metrics": qa1,
        "structural_summary": structural_summary,
        "consistency_summary": consistency_summary,
        "consistency_selectivity": consistency_selectivity,
    }


In [ ]:
SemanticScorer = metric_core.SemanticScorer



def canonical_numeric_atoms(value: object, language: str = "en") -> set[str]:
    normalized = normalize_label(value)
    atoms: set[str] = set()

    # Dates and time-like values are represented component-wise so that
    # 1856-09-22 and 22-09-1856 remain comparable.
    for compound in re.findall(r"(?<!\w)\d+(?:[-/:]\d+)+(?!\w)", normalized):
        atoms.update(str(int(part)) for part in re.findall(r"\d+", compound))
    normalized = re.sub(r"(?<!\w)\d+(?:[-/:]\d+)+(?!\w)", " ", normalized)

    for token in re.findall(r"(?<!\w)\d+(?:[.,]\d+)*(?!\w)", normalized):
        token = token.strip()
        if not token:
            continue
        if "." in token and "," in token:
            decimal = "." if token.rfind(".") > token.rfind(",") else ","
            thousands = "," if decimal == "." else "."
            canonical = token.replace(thousands, "").replace(decimal, ".")
        elif "." in token or "," in token:
            separator = "." if "." in token else ","
            pieces = token.split(separator)
            final_len = len(pieces[-1])
            looks_thousands = (
                len(pieces) > 2
                or (
                    final_len == 3
                    and (
                        (language in {"es", "ca"} and separator == ".")
                        or (language == "en" and separator == ",")
                    )
                )
            )
            canonical = "".join(pieces) if looks_thousands else token.replace(separator, ".")
        else:
            canonical = token
        try:
            parsed = float(canonical)
            atoms.add(str(int(parsed)) if parsed.is_integer() else str(parsed).rstrip("0").rstrip("."))
        except ValueError:
            atoms.add(canonical)

    for token in normalized.split():
        if token in MONTH_NAMES:
            atoms.add(str(MONTH_NAMES[token]))
    return atoms


def literal_retention(
    triples: list[str], text: str, language: str = "en"
) -> tuple[float, int]:
    source_atoms = set()
    for triple in triples:
        subject, predicate, obj = split_triple(triple)
        source_atoms |= canonical_numeric_atoms(subject, language)
        source_atoms |= canonical_numeric_atoms(obj, language)
    if not source_atoms:
        return np.nan, 0
    text_atoms = canonical_numeric_atoms(text, language)
    return len(source_atoms & text_atoms) / len(source_atoms), len(source_atoms)


def literal_atoms_from_triples(
    triples: list[str], language: str = "en"
) -> set[str]:
    atoms: set[str] = set()
    for triple in triples:
        subject, _, obj = split_triple(triple)
        atoms |= canonical_numeric_atoms(subject, language)
        atoms |= canonical_numeric_atoms(obj, language)
    return atoms


def source_target_literal_preservation(
    source_triples: list[str], target_triples: list[str], target_language: str
) -> tuple[float, int]:
    source_atoms = literal_atoms_from_triples(source_triples, "en")
    if not source_atoms:
        return np.nan, 0
    target_atoms = literal_atoms_from_triples(target_triples, target_language)
    return len(source_atoms & target_atoms) / len(source_atoms), len(source_atoms)


def predicate_component_similarity(
    source_triples_column: pd.Series,
    target_triples_column: pd.Series,
    scorer: SemanticScorer,
) -> np.ndarray:
    expanded = []
    for row_index, (source_triples, target_triples) in enumerate(
        zip(source_triples_column, target_triples_column)
    ):
        for source, target in zip(source_triples, target_triples):
            source_predicate = clean_rdf_label(split_triple(source)[1])
            target_predicate = clean_rdf_label(split_triple(target)[1])
            if source_predicate and target_predicate:
                expanded.append(
                    {
                        "row_index": row_index,
                        "source_predicate": source_predicate,
                        "target_predicate": target_predicate,
                    }
                )
    frame = pd.DataFrame(expanded)
    if frame.empty:
        return np.full(len(source_triples_column), np.nan)
    frame["similarity"] = scorer.pairwise(
        frame["source_predicate"], frame["target_predicate"]
    )
    means = frame.groupby("row_index")["similarity"].mean()
    return pd.Series(np.arange(len(source_triples_column))).map(means).to_numpy()


def cluster_bootstrap_mean_ci(
    group: pd.DataFrame,
    value_column: str,
    cluster_column: str = "record_key",
    replicates: int = 1000,
    seed: int = 42,
) -> tuple[float, float]:
    usable = group[[cluster_column, value_column]].dropna()
    clusters = usable[cluster_column].unique()
    if len(clusters) < 2 or replicates <= 0:
        return np.nan, np.nan
    grouped = {
        cluster: usable.loc[usable[cluster_column] == cluster, value_column].to_numpy()
        for cluster in clusters
    }
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(replicates):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        values.append(float(np.concatenate([grouped[item] for item in sampled]).mean()))
    low, high = np.quantile(values, [0.025, 0.975])
    return float(low), float(high)


def min_support(
    triples_column: pd.Series,
    text_column: pd.Series,
    scorer: SemanticScorer,
) -> np.ndarray:
    rows = []
    for row_index, (triples, text) in enumerate(zip(triples_column, text_column)):
        for triple in triples:
            rows.append(
                {
                    "row_index": row_index,
                    "triple": serialize_triple(triple),
                    "text": str(text or ""),
                }
            )
    expanded = pd.DataFrame(rows)
    if expanded.empty:
        return np.full(len(triples_column), np.nan)
    expanded["support"] = scorer.pairwise(expanded["triple"], expanded["text"])
    minima = expanded.groupby("row_index")["support"].min()
    return pd.Series(np.arange(len(triples_column))).map(minima).to_numpy()


QA2_PRIMARY_METRIC = {
    "qa2_wrong_record_text": ("triple_coverage", "decrease"),
    "qa2_entity_substitution_triple": ("triple_similarity", "decrease"),
    "qa2_predicate_substitution_triple": ("predicate_similarity", "decrease"),
    "qa2_structured_predicate_retention": ("structured_predicate_review_flag", "increase"),
    "qa2_entity_substitution_text": ("triple_coverage", "decrease"),
    "qa2_literal_text_only": ("literal_retention", "decrease"),
    "qa2_literal_both": ("source_target_literal_preservation", "decrease"),
    "qa2_omit_one_sentence": ("triple_coverage", "decrease"),
    "qa2_omit_to_first_sentence": ("triple_coverage", "decrease"),
    "qa2_unsupported_addition": ("text_groundedness", "decrease"),
    "qa2_unsupported_addition_plausible": ("text_groundedness", "decrease"),
    "qa2_unsupported_addition_unrelated": ("text_groundedness", "decrease"),
}


def safe_roc_ap(clean: pd.Series, variant: pd.Series, lower_is_worse=True):
    pairs = pd.DataFrame({"clean": clean, "variant": variant}).dropna()
    if len(pairs) < 2:
        return np.nan, np.nan
    y = np.concatenate([np.zeros(len(pairs)), np.ones(len(pairs))])
    values = np.concatenate([pairs["clean"].to_numpy(), pairs["variant"].to_numpy()])
    score = -values if lower_is_worse else values
    if len(np.unique(y)) < 2:
        return np.nan, np.nan
    return roc_auc_score(y, score), average_precision_score(y, score)


In [ ]:
def run_qa2(
    manifest: pd.DataFrame,
    output_dir: Path,
    cache_dir: Path,
    backend: str,
    model_name: str,
    batch_size: int,
    bootstrap_replicates: int,
    force: bool,
) -> dict[str, pd.DataFrame]:
    rows = manifest[
        manifest["qa_track"].isin(["QA2-SEM", "BENIGN-VAR"])
    ].copy().reset_index(drop=True)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_key = hashlib.sha1(
        f"v7|{CORE_COMPATIBILITY}|{backend}|{model_name}|{len(rows)}|"
        f"{rows.variant_id.iloc[0] if len(rows) else ''}".encode()
    ).hexdigest()[:12]
    cache_path = cache_dir / f"qa2_auditstress_metrics_{cache_key}.pkl"

    relative_literal_summary = pd.DataFrame()
    qa22_thresholds = pd.DataFrame()

    if cache_path.exists() and not force:
        metrics = pd.read_pickle(cache_path)
    else:
        if backend != "sentence_transformer":
            raise ValueError(
                "QA2 stress evaluation requires the sentence_transformer "
                "backend defined in Tables QA2.1 and QA2.2."
            )

        scorer = metric_core.SemanticScorer(
            backend,
            model_name,
            batch_size,
        )

        # QA2.1 clean and perturbed metrics.
        clean_qa21 = metric_core.qa21_compute_metrics(
            rows,
            scorer,
            source_text_column="source_text",
            target_text_column="clean_target_text",
            source_triples_column="source_triples",
            target_triples_column="clean_target_triples",
            target_language_column="language",
        ).add_prefix("clean_")
        variant_qa21 = metric_core.qa21_compute_metrics(
            rows,
            scorer,
            source_text_column="source_text",
            target_text_column="variant_target_text",
            source_triples_column="source_triples",
            target_triples_column="variant_target_triples",
            target_language_column="language",
        ).add_prefix("variant_")

        # QA2.2 clean and perturbed instance-level metrics.
        clean_qa22 = metric_core.qa22_compute_metrics(
            rows,
            scorer,
            source_triples_column="source_triples",
            source_text_column="source_text",
            target_triples_column="clean_target_triples",
            target_text_column="clean_target_text",
            target_language_column="language",
        ).add_prefix("clean_")
        variant_qa22 = metric_core.qa22_compute_metrics(
            rows,
            scorer,
            source_triples_column="source_triples",
            source_text_column="source_text",
            target_triples_column="variant_target_triples",
            target_text_column="variant_target_text",
            target_language_column="language",
        ).add_prefix("variant_")

        rows = pd.concat(
            [
                rows.reset_index(drop=True),
                clean_qa21.reset_index(drop=True),
                variant_qa21.reset_index(drop=True),
                clean_qa22.reset_index(drop=True),
                variant_qa22.reset_index(drop=True),
            ],
            axis=1,
        )

        rows["new_structured_predicate_review_flag"] = (
            rows["variant_structured_predicate_review_flag"]
            .fillna(0).astype(bool)
            & ~rows["clean_structured_predicate_review_flag"]
            .fillna(0).astype(bool)
        ).astype(float)

        # Auxiliary selectivity diagnostic, not a QA2 table metric.
        rows["clean_source_identical_predicate_review"] = [
            float(metric_core.source_identical_predicate(source, target))
            for source, target in zip(
                rows["source_triples"],
                rows["clean_target_triples"],
            )
        ]
        rows["variant_source_identical_predicate_review"] = [
            float(metric_core.source_identical_predicate(source, target))
            for source, target in zip(
                rows["source_triples"],
                rows["variant_target_triples"],
            )
        ]
        rows["new_source_identical_predicate_review"] = (
            rows["variant_source_identical_predicate_review"].astype(bool)
            & ~rows["clean_source_identical_predicate_review"].astype(bool)
        ).astype(float)

        # Exact-size English calibration. Each base English lexicalisation is
        # used once, despite appearing in multiple benchmark variants.
        english_base = rows[[
            "base_pair_key",
            "source_triples",
            "source_text",
        ]].drop_duplicates("base_pair_key").reset_index(drop=True)
        english_calibration, qa22_thresholds = (
            metric_core.qa22_build_english_calibration(
                english_base,
                scorer,
                triples_column="source_triples",
                text_column="source_text",
                percentile=0.05,
            )
        )

        clean_calibrated = metric_core.qa22_apply_english_calibration(
            rows[[
                "clean_target_triple_count",
                "clean_minimum_triple_coverage",
                "clean_minimum_text_groundedness",
            ]].rename(columns={
                "clean_target_triple_count": "target_triple_count",
                "clean_minimum_triple_coverage": "minimum_triple_coverage",
                "clean_minimum_text_groundedness": "minimum_text_groundedness",
            }),
            qa22_thresholds,
        ).add_prefix("clean_")
        variant_calibrated = metric_core.qa22_apply_english_calibration(
            rows[[
                "variant_target_triple_count",
                "variant_minimum_triple_coverage",
                "variant_minimum_text_groundedness",
            ]].rename(columns={
                "variant_target_triple_count": "target_triple_count",
                "variant_minimum_triple_coverage": "minimum_triple_coverage",
                "variant_minimum_text_groundedness": "minimum_text_groundedness",
            }),
            qa22_thresholds,
        ).add_prefix("variant_")

        # Only append calibration outputs that are not already present.
        for calibrated in [clean_calibrated, variant_calibrated]:
            for column in calibrated.columns:
                if column not in rows.columns:
                    rows[column] = calibrated[column].to_numpy()

        qa32_benign = rows["qa_track"].eq("BENIGN-VAR")
        qa32_aligned_source_text = rows["source_text"].where(
            ~qa32_benign,
            rows["alternative_source_text"],
        )
        qa32_target_lid = rows["source_lid"].where(
            ~qa32_benign,
            rows["alternative_reference_lid"],
        )
        qa32_comparison_text = rows["source_text"].where(
            qa32_benign,
            "",
        )
        qa32_comparison_lid = rows["source_lid"].where(
            qa32_benign,
            "",
        )

        qa32_metrics = metric_core.qa32_compute_metrics(
            aligned_source_texts=qa32_aligned_source_text,
            target_texts=rows["variant_target_text"],
            comparison_source_texts=qa32_comparison_text,
            scorer=scorer,
            target_lids=qa32_target_lid,
            aligned_source_lids=qa32_target_lid,
            comparison_source_lids=qa32_comparison_lid,
            target_record_keys=rows["record_key"],
            aligned_source_record_keys=rows["record_key"],
            comparison_source_record_keys=rows["record_key"],
        )
        rows["variant_aligned_reference_similarity"] = qa32_metrics[
            "aligned_reference_similarity"
        ].to_numpy()
        rows["variant_comparison_reference_similarity"] = qa32_metrics[
            "comparison_reference_similarity"
        ].to_numpy()
        rows["aligned_reference_advantage"] = qa32_metrics[
            "aligned_reference_advantage"
        ].to_numpy()
        rows["aligned_reference_eligible"] = qa32_metrics[
            "aligned_reference_eligible"
        ].to_numpy()

        # Backward-compatible names used by the stress-detection tables.
        for prefix in ["clean", "variant"]:
            rows[f"{prefix}_triple_coverage"] = rows[
                f"{prefix}_minimum_triple_coverage"
            ]
            rows[f"{prefix}_text_groundedness"] = rows[
                f"{prefix}_minimum_text_groundedness"
            ]
            rows[f"{prefix}_literal_retention"] = rows[
                f"{prefix}_target_text_literal_retention"
            ]

        for metric in [
            "text_similarity",
            "triple_similarity",
            "predicate_similarity",
            "structured_predicate_review_flag",
            "triple_coverage",
            "text_groundedness",
            "literal_retention",
            "source_target_literal_preservation",
        ]:
            rows[f"delta_{metric}"] = (
                rows[f"variant_{metric}"]
                - rows[f"clean_{metric}"]
            )

        # Shared-table contract checks on the generated stress rows.
        for prefix in ["clean_", "variant_"]:
            for metric_name in [
                "text_similarity",
                "triple_similarity",
                "predicate_similarity",
                "minimum_triple_coverage",
                "minimum_text_groundedness",
            ]:
                values = rows[f"{prefix}{metric_name}"].dropna()
                if not values.between(
                    -1.0, 1.0, inclusive="both"
                ).all():
                    raise AssertionError(
                        f"{prefix}{metric_name} must lie in [-1, 1]."
                    )

            for metric_name in [
                "source_target_literal_preservation",
                "target_text_literal_retention",
            ]:
                values = rows[f"{prefix}{metric_name}"].dropna()
                if not values.between(
                    0.0, 1.0, inclusive="both"
                ).all():
                    raise AssertionError(
                        f"{prefix}{metric_name} must lie in [0, 1]."
                    )

            invalid = ~rows[f"{prefix}predicate_alignment_valid"]
            if rows.loc[
                invalid,
                f"{prefix}predicate_similarity",
            ].notna().any():
                raise AssertionError(
                    "PredSim must be undefined for incomplete alignment."
                )

            threshold_missing = rows[
                f"{prefix}coverage_threshold"
            ].isna()
            if rows.loc[
                threshold_missing,
                f"{prefix}english_calibrated_coverage_pass",
            ].notna().any():
                raise AssertionError(
                    "QA2.2 coverage used a non-exact fallback threshold."
                )

        # R_lit is a language-level statistic. Report clean and perturbed
        # benchmark values separately rather than treating it as a row metric.
        clean_relative_input = rows[[
            "language",
            "language_label",
            "clean_english_literal_eligible",
            "clean_target_literal_full_success",
        ]].rename(columns={
            "clean_english_literal_eligible": "english_literal_eligible",
            "clean_target_literal_full_success": "target_literal_full_success",
        })
        variant_relative_input = rows[[
            "language",
            "language_label",
            "variant_english_literal_eligible",
            "variant_target_literal_full_success",
        ]].rename(columns={
            "variant_english_literal_eligible": "english_literal_eligible",
            "variant_target_literal_full_success": "target_literal_full_success",
        })
        clean_relative = (
            metric_core.qa22_relative_literal_retention_summary(
                clean_relative_input
            ).rename(columns={
                "english_eligible_instances": "clean_english_eligible_instances",
                "target_successes_among_english_eligible": (
                    "clean_target_successes_among_english_eligible"
                ),
                "relative_literal_retention": (
                    "clean_relative_literal_retention"
                ),
            })
        )
        variant_relative = (
            metric_core.qa22_relative_literal_retention_summary(
                variant_relative_input
            ).rename(columns={
                "english_eligible_instances": "variant_english_eligible_instances",
                "target_successes_among_english_eligible": (
                    "variant_target_successes_among_english_eligible"
                ),
                "relative_literal_retention": (
                    "variant_relative_literal_retention"
                ),
            })
        )
        relative_literal_summary = clean_relative.merge(
            variant_relative,
            on=["language", "language_label", "audited_instances"],
            how="outer",
            validate="one_to_one",
        )
        relative_literal_summary["delta_relative_literal_retention"] = (
            relative_literal_summary[
                "variant_relative_literal_retention"
            ]
            - relative_literal_summary[
                "clean_relative_literal_retention"
            ]
        )

        rows.to_pickle(cache_path)
        metrics = rows
    
    # Reconstruct exact-size thresholds when loading cached row metrics.
    if qa22_thresholds.empty and not metrics.empty:
        clean_thresholds = metrics[[
            "clean_target_triple_count",
            "clean_coverage_threshold",
            "clean_groundedness_threshold",
        ]].rename(columns={
            "clean_target_triple_count": "triple_count",
            "clean_coverage_threshold": "coverage_threshold",
            "clean_groundedness_threshold": "groundedness_threshold",
        })
        variant_thresholds = metrics[[
            "variant_target_triple_count",
            "variant_coverage_threshold",
            "variant_groundedness_threshold",
        ]].rename(columns={
            "variant_target_triple_count": "triple_count",
            "variant_coverage_threshold": "coverage_threshold",
            "variant_groundedness_threshold": "groundedness_threshold",
        })
        qa22_thresholds = (
            pd.concat([clean_thresholds, variant_thresholds], ignore_index=True)
            .dropna(subset=[
                "coverage_threshold",
                "groundedness_threshold",
            ], how="all")
            .drop_duplicates("triple_count")
            .sort_values("triple_count")
            .reset_index(drop=True)
        )

    # Reconstruct aggregate QA2.2 outputs when loading cached row metrics.
    if relative_literal_summary.empty and not metrics.empty:
        clean_relative_input = metrics[[
            "language", "language_label",
            "clean_english_literal_eligible",
            "clean_target_literal_full_success",
        ]].rename(columns={
            "clean_english_literal_eligible": "english_literal_eligible",
            "clean_target_literal_full_success": "target_literal_full_success",
        })
        variant_relative_input = metrics[[
            "language", "language_label",
            "variant_english_literal_eligible",
            "variant_target_literal_full_success",
        ]].rename(columns={
            "variant_english_literal_eligible": "english_literal_eligible",
            "variant_target_literal_full_success": "target_literal_full_success",
        })
        clean_relative = metric_core.qa22_relative_literal_retention_summary(
            clean_relative_input
        ).rename(columns={
            "english_eligible_instances": "clean_english_eligible_instances",
            "target_successes_among_english_eligible": (
                "clean_target_successes_among_english_eligible"
            ),
            "relative_literal_retention": "clean_relative_literal_retention",
        })
        variant_relative = metric_core.qa22_relative_literal_retention_summary(
            variant_relative_input
        ).rename(columns={
            "english_eligible_instances": "variant_english_eligible_instances",
            "target_successes_among_english_eligible": (
                "variant_target_successes_among_english_eligible"
            ),
            "relative_literal_retention": "variant_relative_literal_retention",
        })
        relative_literal_summary = clean_relative.merge(
            variant_relative,
            on=["language", "language_label", "audited_instances"],
            how="outer",
            validate="one_to_one",
        )
        relative_literal_summary["delta_relative_literal_retention"] = (
            relative_literal_summary["variant_relative_literal_retention"]
            - relative_literal_summary["clean_relative_literal_retention"]
        )

    summary_rows = []
    corruption_metrics = metrics[metrics["qa_track"].eq("QA2-SEM")]
    for (language, corruption, severity), group in corruption_metrics.groupby([
        "language", "corruption_type", "severity"
    ]):
        metric, direction = QA2_PRIMARY_METRIC.get(
            corruption,
            ("text_similarity", "decrease"),
        )
        if corruption == "qa2_structured_predicate_retention":
            detected = group[
                "new_structured_predicate_review_flag"
            ].astype(bool)
            summary_rows.append({
                "language": language,
                "language_label": LANG_LABELS[language],
                "corruption_type": corruption,
                "severity": severity,
                "variants": len(group),
                "primary_metric": metric,
                "expected_direction": "increase/flag",
                "clean_mean": group[
                    "clean_structured_predicate_review_flag"
                ].mean(),
                "variant_mean": group[
                    "variant_structured_predicate_review_flag"
                ].mean(),
                "mean_delta": group[
                    "delta_structured_predicate_review_flag"
                ].mean(),
                "median_delta": group[
                    "delta_structured_predicate_review_flag"
                ].median(),
                "cluster_bootstrap_ci_low": np.nan,
                "cluster_bootstrap_ci_high": np.nan,
                "bootstrap_replicates": bootstrap_replicates,
                "expected_direction_rate": detected.mean(),
                "auroc_clean_vs_variant": np.nan,
                "auprc_clean_vs_variant": np.nan,
            })
            continue
        delta = group[f"delta_{metric}"]
        success = delta < 0 if direction == "decrease" else delta > 0
        roc, ap = safe_roc_ap(
            group[f"clean_{metric}"],
            group[f"variant_{metric}"],
            lower_is_worse=(direction == "decrease"),
        )
        ci_low, ci_high = cluster_bootstrap_mean_ci(
            group,
            f"delta_{metric}",
            replicates=bootstrap_replicates,
            seed=RANDOM_SEED + stable_hash_int(language, corruption),
        )
        summary_rows.append({
            "language": language,
            "language_label": LANG_LABELS[language],
            "corruption_type": corruption,
            "severity": severity,
            "variants": len(group),
            "primary_metric": metric,
            "expected_direction": direction,
            "clean_mean": group[f"clean_{metric}"].mean(),
            "variant_mean": group[f"variant_{metric}"].mean(),
            "mean_delta": delta.mean(),
            "median_delta": delta.median(),
            "cluster_bootstrap_ci_low": ci_low,
            "cluster_bootstrap_ci_high": ci_high,
            "bootstrap_replicates": bootstrap_replicates,
            "expected_direction_rate": success.mean(),
            "auroc_clean_vs_variant": roc,
            "auprc_clean_vs_variant": ap,
        })
    summary = pd.DataFrame(summary_rows)

    benign = metrics[metrics["qa_track"].eq("BENIGN-VAR")].copy()
    benign_summary = (
        benign.groupby(["language", "language_label"])
        .agg(
            variants=("variant_id", "size"),
            mean_aligned_reference_advantage=(
                "aligned_reference_advantage", "mean"
            ),
            positive_aligned_reference_advantage_rate=(
                "aligned_reference_advantage",
                lambda s: (s.dropna() > 0).mean(),
            ),
            text_similarity_delta=("delta_text_similarity", "mean"),
            coverage_delta=("delta_triple_coverage", "mean"),
            groundedness_delta=("delta_text_groundedness", "mean"),
            text_similarity_non_decrease_rate=(
                "delta_text_similarity", lambda s: (s >= -0.05).mean()
            ),
            coverage_non_decrease_rate=(
                "delta_triple_coverage", lambda s: (s >= -0.05).mean()
            ),
            groundedness_non_decrease_rate=(
                "delta_text_groundedness", lambda s: (s >= -0.05).mean()
            ),
            broad_source_identical_review_rate=(
                "variant_source_identical_predicate_review", "mean"
            ),
            new_source_identical_review_rate=(
                "new_source_identical_predicate_review", "mean"
            ),
        ).reset_index()
    )

    omission = metrics[
        metrics["corruption_type"].isin([
            "qa2_omit_one_sentence",
            "qa2_omit_to_first_sentence",
        ])
        & metrics["perturbation_pair_id"].fillna("").ne("")
    ].copy()
    omission_paired = omission.pivot_table(
        index=[
            "language", "language_label", "record_key",
            "perturbation_pair_id",
        ],
        columns="severity",
        values=[
            "variant_triple_coverage",
            "variant_text_similarity",
            "removed_token_fraction",
        ],
        aggfunc="first",
    )
    omission_paired.columns = [
        "_".join(map(str, c)) for c in omission_paired.columns
    ]
    omission_paired = omission_paired.reset_index()
    omission_severity = (
        omission_paired.groupby(["language", "language_label"])
        .apply(lambda g: pd.Series({
            "paired_instances": len(g),
            "severe_lower_coverage_rate": (
                g["variant_triple_coverage_severe"]
                < g["variant_triple_coverage_mild"]
            ).mean(),
            "severe_lower_text_similarity_rate": (
                g["variant_text_similarity_severe"]
                < g["variant_text_similarity_mild"]
            ).mean(),
        }), include_groups=False)
        .reset_index()
    ) if not omission_paired.empty else pd.DataFrame()

    additions = metrics[
        metrics["corruption_type"].isin([
            "qa2_unsupported_addition_plausible",
            "qa2_unsupported_addition_unrelated",
        ])
        & metrics["perturbation_pair_id"].fillna("").ne("")
    ].copy()
    addition_paired = additions.pivot_table(
        index=[
            "language", "language_label", "record_key",
            "perturbation_pair_id",
        ],
        columns="severity",
        values=[
            "variant_text_groundedness",
            "variant_text_similarity",
        ],
        aggfunc="first",
    )
    addition_paired.columns = [
        "_".join(map(str, c)) for c in addition_paired.columns
    ]
    addition_paired = addition_paired.reset_index()
    addition_severity = (
        addition_paired.groupby(["language", "language_label"])
        .apply(lambda g: pd.Series({
            "paired_instances": len(g),
            "unrelated_lower_groundedness_rate": (
                g["variant_text_groundedness_severe"]
                < g["variant_text_groundedness_mild"]
            ).mean(),
            "unrelated_lower_similarity_rate": (
                g["variant_text_similarity_severe"]
                < g["variant_text_similarity_mild"]
            ).mean(),
        }), include_groups=False)
        .reset_index()
    ) if not addition_paired.empty else pd.DataFrame()

    save_table(metrics, output_dir / "qa2_variant_metrics.csv")
    save_table(summary, output_dir / "qa2_corruption_detection_summary.csv")
    save_table(benign_summary, output_dir / "qa2_benign_variation_summary.csv")
    save_table(omission_severity, output_dir / "qa2_omission_severity_summary.csv")
    save_table(addition_severity, output_dir / "qa2_addition_severity_summary.csv")
    save_table(
        relative_literal_summary,
        output_dir / "qa22_relative_literal_retention_summary.csv",
    )
    if not qa22_thresholds.empty:
        save_table(
            qa22_thresholds,
            output_dir / "qa22_english_calibration_thresholds.csv",
        )

    return {
        "variant_metrics": metrics,
        "summary": summary,
        "benign_summary": benign_summary,
        "omission_severity": omission_severity,
        "addition_severity": addition_severity,
        "relative_literal_summary": relative_literal_summary,
        "qa22_thresholds": qa22_thresholds,
    }


In [ ]:


# ---------------------------------------------------------------------------
# QA3
# ---------------------------------------------------------------------------

PLACEHOLDER_RE = re.compile(
    r"^(?:none|null|nan|n/?a|unknown|undefined)$", re.IGNORECASE
)
SOURCE_MARKUP_RE = re.compile(r"(?:@[Ee][Nn]\b|\^\^)")
VALID_URI_RE = re.compile(r"^(?:<https?://[^>]+>|https?://\S+)$", re.IGNORECASE)
CAMEL_CASE_RE = re.compile(r"[a-zà-ÿ][A-Z]")


def malformed_component(value: str) -> bool:
    text = normalize_space(value)
    if VALID_URI_RE.fullmatch(text):
        return False
    return (
        not text
        or bool(PLACEHOLDER_RE.fullmatch(text))
        or bool(SOURCE_MARKUP_RE.search(text))
    )


def triple_well_formed(triples: list[str]) -> bool:
    if not triples:
        return False
    for triple in triples:
        components = split_triple(triple)
        if any(malformed_component(component) for component in components):
            return False
    return True


def source_identical_predicate(
    source_triples: list[str], target_triples: list[str]
) -> bool:
    return any(
        normalize_space(split_triple(source)[1])
        == normalize_space(split_triple(target)[1])
        for source, target in zip(source_triples, target_triples)
    )


def structured_untranslated_predicate(
    source_triples: list[str], target_triples: list[str]
) -> bool:
    for source_triple, target_triple in zip(source_triples, target_triples):
        source_predicate = split_triple(source_triple)[1]
        target_predicate = split_triple(target_triple)[1]
        if (
            normalize_space(source_predicate) == normalize_space(target_predicate)
            and bool(CAMEL_CASE_RE.search(source_predicate))
        ):
            return True
    return False


def parse_original_language_data(repo_root: Path, max_per_language: int):
    data_root = repo_root / "WebNLG_CA_BT"
    if not data_root.exists():
        raise FileNotFoundError(f"Original corpus not found at {data_root}")

    rows = []
    for path in sorted(data_root.rglob("*.xml")):
        relative = path.relative_to(data_root)
        if any(part.startswith(".") for part in relative.parts):
            continue
        if path.name.endswith("-checkpoint.xml"):
            continue
        try:
            root = ET.parse(path).getroot()
        except ET.ParseError:
            continue
        for entry in root.findall(".//entry"):
            eid = (entry.get("eid") or "").strip()
            record_key = f"{relative.as_posix()}::{eid}"
            lex = defaultdict(dict)
            for node in entry.findall("lex"):
                language = (node.get("lang") or "").strip()
                lid = (node.get("lid") or "").strip()
                text = normalize_space(node.text)
                if language and lid and text:
                    lex[language][lid] = text
            for language in ("en", "es", "ca"):
                for lid, text in lex.get(language, {}).items():
                    rows.append(
                        {
                            "record_key": record_key,
                            "language": language,
                            "text": text,
                        }
                    )
    frame = pd.DataFrame(rows).drop_duplicates()
    sampled = []
    for language, group in frame.groupby("language"):
        n = min(max_per_language, len(group))
        sampled.append(group.sample(n=n, random_state=RANDOM_SEED))
    return pd.concat(sampled, ignore_index=True)


def train_language_detector(
    repo_root: Path,
    cache_dir: Path,
    max_per_language: int,
    force: bool,
):
    model_path = cache_dir / f"qa3_language_detector_{max_per_language}.joblib"
    metrics_path = cache_dir / f"qa3_language_detector_{max_per_language}.json"
    if (
        joblib is not None
        and model_path.exists()
        and metrics_path.exists()
        and not force
    ):
        model = joblib.load(model_path)
        metadata = json.loads(metrics_path.read_text(encoding="utf-8"))
        return model, metadata

    data = parse_original_language_data(repo_root, max_per_language)
    splitter = GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=RANDOM_SEED
    )
    train_idx, test_idx = next(
        splitter.split(data["text"], data["language"], groups=data["record_key"])
    )
    train = data.iloc[train_idx]
    test = data.iloc[test_idx]

    model = Pipeline(
        [
            (
                "vectorizer",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    max_features=60_000,
                    sublinear_tf=True,
                    dtype=np.float32,
                ),
            ),
            (
                "classifier",
                SGDClassifier(
                    loss="log_loss",
                    max_iter=50,
                    tol=1e-3,
                    class_weight="balanced",
                    average=True,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )
    model.fit(train["text"], train["language"])
    prediction = model.predict(test["text"])
    metadata = {
        "grouped_holdout_accuracy": float(
            accuracy_score(test["language"], prediction)
        ),
        "training_examples": int(len(train)),
        "holdout_examples": int(len(test)),
        "max_per_language": int(max_per_language),
    }
    if joblib is not None:
        cache_dir.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, model_path)
        metrics_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return model, metadata


def predict_language(model, texts: pd.Series):
    texts = texts.fillna("").astype(str)
    probabilities = model.predict_proba(texts)
    classes = model.named_steps["classifier"].classes_
    index = probabilities.argmax(axis=1)
    predicted = classes[index]
    confidence = probabilities.max(axis=1)
    class_position = {label: i for i, label in enumerate(classes)}
    return predicted, confidence, probabilities, class_position


def code_switch_flag(model, text: str, expected_language: str) -> bool:
    tokens = normalize_space(text).split()
    if len(tokens) < 8:
        return False
    windows = [
        " ".join(tokens[start : start + 10])
        for start in range(0, len(tokens), 8)
        if len(tokens[start : start + 10]) >= 6
    ]
    if not windows:
        return False
    probabilities = model.predict_proba(windows)
    classes = model.named_steps["classifier"].classes_
    english_index = list(classes).index("en")
    english_windows = probabilities[:, english_index] >= 0.70
    return float(english_windows.mean()) >= 0.25


def internal_chrf(reference: str, hypothesis: str, max_order: int = 6) -> float:
    reference = normalize_space(reference)
    hypothesis = normalize_space(hypothesis)
    if not reference or not hypothesis:
        return 0.0
    scores = []
    for order in range(1, max_order + 1):
        ref = defaultdict(int)
        hyp = defaultdict(int)
        for i in range(max(0, len(reference) - order + 1)):
            ref[reference[i : i + order]] += 1
        for i in range(max(0, len(hypothesis) - order + 1)):
            hyp[hypothesis[i : i + order]] += 1
        overlap = sum(min(count, hyp[gram]) for gram, count in ref.items())
        precision = overlap / max(1, sum(hyp.values()))
        recall = overlap / max(1, sum(ref.values()))
        scores.append(
            0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        )
    return float(np.mean(scores))


In [ ]:
QA3_DETECTOR = {
    "qa3_full_english_copy": "full_english_copy_detected",
    "qa3_english_clause_insertion": "code_switch_detected",
}


def run_qa3(
    manifest: pd.DataFrame,
    repo_root: Path,
    output_dir: Path,
    cache_dir: Path,
    language_training_max: int,
    semantic_backend: str,
    semantic_model: str,
    semantic_batch_size: int,
    force: bool,
) -> dict[str, pd.DataFrame]:
    rows = manifest[
        manifest["qa_track"].isin(["CLEAN", "QA3-LANG", "BENIGN-VAR"])
    ].copy().reset_index(drop=True)

    model, detector_metadata = metric_core.train_language_detector(
        repo_root,
        cache_dir,
        language_training_max,
        force,
    )
    qa31_metrics = metric_core.qa31_compute_metrics(
        target_texts=rows["variant_target_text"],
        source_texts=rows["source_text"],
        expected_languages=rows["language"],
        model=model,
    )
    for column in qa31_metrics.columns:
        rows[column] = qa31_metrics[column].to_numpy()

    # Dataset-level checks complement the synthetic QA3.1 contract.
    expected_valid = (
        rows["predicted_language"].eq(rows["language"])
        & rows["target_language_probability"].ge(
            metric_core.QA31_LANGUAGE_CONFIDENCE
        )
    )
    if not rows["valid_target_language"].eq(
        expected_valid
    ).all():
        raise AssertionError(
            "AuditStress V_l(e) does not match the QA3.1 rule."
        )

    expected_wrong = (
        rows["predicted_language"].ne(rows["language"])
        & rows["language_confidence"].ge(
            metric_core.QA31_LANGUAGE_CONFIDENCE
        )
    )
    if not rows["wrong_language_detected"].eq(
        expected_wrong
    ).all():
        raise AssertionError(
            "AuditStress W_l(e) does not match the QA3.1 rule."
        )
    rows["hard_language_flag"] = rows[[
        "wrong_language_detected", "full_english_copy_detected", "code_switch_detected"
    ]].any(axis=1)

    has_qa32_pairs = rows["qa_track"].eq("BENIGN-VAR").any()
    if has_qa32_pairs and semantic_backend != "sentence_transformer":
        raise ValueError(
            "QA3.2 aligned-reference advantage requires the "
            "sentence_transformer backend used for multilingual "
            "semantic similarity."
        )

    scorer = metric_core.SemanticScorer(
        semantic_backend,
        semantic_model,
        semantic_batch_size,
    )

    empty_text = pd.Series("", index=rows.index, dtype="string")
    empty_lid = pd.Series("", index=rows.index, dtype="string")

    clean_qa32 = metric_core.qa32_compute_metrics(
        aligned_source_texts=rows["source_text"],
        target_texts=rows["clean_target_text"],
        comparison_source_texts=empty_text,
        scorer=scorer,
        target_lids=rows["source_lid"],
        aligned_source_lids=rows["source_lid"],
        comparison_source_lids=empty_lid,
        target_record_keys=rows["record_key"],
        aligned_source_record_keys=rows["record_key"],
        comparison_source_record_keys=rows["record_key"],
    )

    benign_mask = rows["qa_track"].eq("BENIGN-VAR")
    variant_aligned_source_text = rows["source_text"].where(
        ~benign_mask,
        rows["alternative_source_text"],
    )
    variant_target_lid = rows["source_lid"].where(
        ~benign_mask,
        rows["alternative_reference_lid"],
    )
    variant_comparison_text = rows["source_text"].where(
        benign_mask,
        "",
    )
    variant_comparison_lid = rows["source_lid"].where(
        benign_mask,
        "",
    )

    variant_qa32 = metric_core.qa32_compute_metrics(
        aligned_source_texts=variant_aligned_source_text,
        target_texts=rows["variant_target_text"],
        comparison_source_texts=variant_comparison_text,
        scorer=scorer,
        target_lids=variant_target_lid,
        aligned_source_lids=variant_target_lid,
        comparison_source_lids=variant_comparison_lid,
        target_record_keys=rows["record_key"],
        aligned_source_record_keys=rows["record_key"],
        comparison_source_record_keys=rows["record_key"],
    )

    for column in clean_qa32.columns:
        rows[f"clean_{column}"] = clean_qa32[column].to_numpy()
        rows[f"variant_{column}"] = variant_qa32[column].to_numpy()

    # Backward-compatible unprefixed names used by existing summaries.
    rows["aligned_reference_similarity"] = rows[
        "variant_aligned_reference_similarity"
    ]
    rows["comparison_reference_similarity"] = rows[
        "variant_comparison_reference_similarity"
    ]
    rows["aligned_reference_advantage"] = rows[
        "variant_aligned_reference_advantage"
    ]
    rows["aligned_reference_eligible"] = rows[
        "variant_aligned_reference_eligible"
    ]

    # Dataset-level QA3.2 checks.
    for prefix in ["clean", "variant"]:
        if not rows[f"{prefix}_expansion_ratio"].ge(0.0).all():
            raise AssertionError(
                f"{prefix} QA3.2 expansion ratios must be non-negative."
            )

    if not rows.loc[
        benign_mask,
        "variant_aligned_reference_eligible",
    ].all():
        raise AssertionError(
            "Every AuditStress benign alternative must form a valid "
            "same-entry a-versus-b QA3.2 comparison."
        )
    if rows.loc[
        ~benign_mask,
        "variant_aligned_reference_advantage",
    ].notna().any():
        raise AssertionError(
            "QA3.2 aligned-reference advantage must remain undefined "
            "outside the benign multi-reference track."
        )

    rows["expected_detector"] = rows["corruption_type"].map(QA3_DETECTOR)
    rows["detected"] = False
    for detector in QA3_DETECTOR.values():
        mask = rows["expected_detector"].eq(detector)
        rows.loc[mask, "detected"] = rows.loc[mask, detector]

    summary = (
        rows[rows["qa_track"].eq("QA3-LANG")]
        .groupby(["language", "language_label", "corruption_type", "severity"], dropna=False)
        .agg(
            variants=("variant_id", "size"),
            detection_rate=("detected", "mean"),
            valid_target_language_rate=("valid_target_language", "mean"),
            wrong_language_rate=("wrong_language_detected", "mean"),
            exact_copy_rate=("full_english_copy_detected", "mean"),
            code_switch_rate=("code_switch_detected", "mean"),
            mean_target_language_probability=("target_language_probability", "mean"),
        ).reset_index()
    )

    controls = rows[rows["qa_track"].isin(["CLEAN", "BENIGN-VAR"])].copy()
    control_selectivity = (
        controls.groupby(["qa_track", "language", "language_label"])
        .agg(
            variants=("variant_id", "size"),
            hard_flag_false_positive_rate=("hard_language_flag", "mean"),
            valid_target_language_rate=("valid_target_language", "mean"),
            wrong_language_rate=("wrong_language_detected", "mean"),
            exact_copy_rate=("full_english_copy_detected", "mean"),
            code_switch_rate=("code_switch_detected", "mean"),
        ).reset_index()
    )

    benign = rows[rows["qa_track"].eq("BENIGN-VAR")].copy()
    if not benign.empty:
        benign["expansion_delta"] = (
            benign["variant_expansion_ratio"]
            - benign["clean_expansion_ratio"]
        )
        benign_summary = (
            benign.groupby(["language", "language_label"])
            .agg(
                variants=("variant_id", "size"),
                mean_expansion_change=("expansion_delta", "mean"),
                mean_aligned_reference_advantage=(
                    "aligned_reference_advantage", "mean"
                ),
                positive_aligned_reference_advantage_rate=(
                    "aligned_reference_advantage",
                    lambda values: (values.dropna() > 0).mean(),
                ),
            ).reset_index()
        )
    else:
        benign_summary = pd.DataFrame()

    save_table(rows, output_dir / "qa3_variant_metrics.csv")
    save_table(summary, output_dir / "qa3_corruption_detection_summary.csv")
    save_table(benign_summary, output_dir / "qa3_benign_variation_summary.csv")
    save_table(control_selectivity, output_dir / "qa3_control_selectivity_summary.csv")
    (output_dir / "qa3_language_detector_metadata.json").write_text(
        json.dumps(detector_metadata, indent=2), encoding="utf-8"
    )
    qa3_metric_metadata = {
        "metric_core_compatibility": metric_core.CORE_COMPATIBILITY,
        "qa31_shared_metric_core": True,
        "qa32_shared_metric_core": True,
        "qa32_tokenisation": "whitespace tokens after normalize_space",
        "qa32_advantage_scope": (
            "benign alternative target lexicalisation a, its aligned "
            "English lexicalisation a, and the original different "
            "English lexicalisation b from the same RDF entry"
        ),
        "qa32_advantage_defined_only_for_nonempty_same_entry_a_not_equal_b": True,
    }
    (output_dir / "qa3_metric_metadata.json").write_text(
        json.dumps(qa3_metric_metadata, indent=2),
        encoding="utf-8",
    )

    return {
        "variant_metrics": rows,
        "summary": summary,
        "benign_summary": benign_summary,
        "control_selectivity": control_selectivity,
    }


In [ ]:



# ---------------------------------------------------------------------------
# Shared stress-aligned metric core
# ---------------------------------------------------------------------------
# Imported after the local definitions so all run functions resolve the
# canonical shared implementations loaded from CORE_PATH. The corrected
# 01_QA1 notebook imports the same physical module file.
from audit_metric_core import (
    CORE_COMPATIBILITY,
    normalize_space,
    normalize_label,
    split_triple,
    clean_rdf_label,
    serialize_triple,
    serialize_triple_set,
    SemanticScorer,
    canonical_numeric_atoms,
    literal_retention,
    literal_atoms_from_triples,
    source_target_literal_preservation,
    predicate_component_similarity,
    predicate_component_diagnostics,
    qa21_pairwise_similarity,
    qa21_compute_metrics,
    assert_qa21_metric_contract,
    qa22_compute_metrics,
    qa22_build_english_calibration,
    qa22_apply_english_calibration,
    qa22_relative_literal_retention_summary,
    assert_qa22_metric_contract,
    cluster_bootstrap_mean_ci,
    min_support,
    minimum_triple_coverage,
    textual_units,
    minimum_text_groundedness,
    aligned_reference_advantage,
    malformed_component,
    triple_well_formed,
    source_identical_predicate,
    structured_untranslated_predicate,
    qa12_profile_normalized_forms,
    qa12_build_mapping_profiles,
    qa12_summarize_mapping_profiles,
    assert_qa12_metric_contract,
    parse_original_language_data,
    train_language_detector,
    predict_language,
    qa31_eligible_windows,
    qa31_code_switch_profile,
    code_switch_flag,
    qa31_compute_metrics,
    assert_qa31_metric_contract,
    qa32_tokens,
    qa32_select_comparison_references,
    qa32_compute_metrics,
    assert_qa32_metric_contract,
)


def build_combined_summary(
    qa1_results: dict | None,
    qa2_results: dict | None,
    qa3_results: dict | None,
    output_dir: Path,
) -> pd.DataFrame:
    frames = []
    if qa1_results is not None:
        frame = qa1_results["structural_summary"].copy()
        frame = frame[frame["qa_track"].eq("QA1-STRUCT")].copy()
        frame["audit_dimension"] = "QA1.1"
        frame["primary_detection_rate"] = frame["detection_rate"]
        frames.append(
            frame[
                [
                    "audit_dimension", "language", "language_label",
                    "corruption_type", "severity", "variants",
                    "primary_detection_rate",
                ]
            ]
        )
        consistency = qa1_results["consistency_summary"].copy()
        if not consistency.empty:
            consistency = consistency[
                consistency["group_kind"].eq("controlled_harmful_inconsistency")
            ].copy()
            consistency["audit_dimension"] = "QA1.2"
            consistency["corruption_type"] = "qa1_mapping_inconsistency"
            consistency["severity"] = "mild"
            consistency["variants"] = consistency["occurrences"]
            consistency["primary_detection_rate"] = consistency[
                "inconsistency_detected"
            ].astype(float)
            frames.append(
                consistency[
                    [
                        "audit_dimension", "language", "language_label",
                        "corruption_type", "severity", "variants",
                        "primary_detection_rate",
                    ]
                ]
            )
    if qa2_results is not None:
        frame = qa2_results["summary"].copy()
        frame["audit_dimension"] = "QA2"
        frame["primary_detection_rate"] = frame["expected_direction_rate"]
        frames.append(
            frame[
                [
                    "audit_dimension", "language", "language_label",
                    "corruption_type", "severity", "variants",
                    "primary_detection_rate",
                ]
            ]
        )
    if qa3_results is not None:
        frame = qa3_results["summary"].copy()
        frame["audit_dimension"] = "QA3.1"
        frame["primary_detection_rate"] = frame["detection_rate"]
        frames.append(
            frame[
                [
                    "audit_dimension", "language", "language_label",
                    "corruption_type", "severity", "variants",
                    "primary_detection_rate",
                ]
            ]
        )

    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    save_table(combined, output_dir / "combined_detection_summary.csv")
    return combined


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo-root", type=Path, required=True)
    parser.add_argument("--benchmark-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument(
        "--tracks",
        nargs="+",
        choices=["qa1", "qa2", "qa3"],
        default=["qa1", "qa2", "qa3"],
    )
    parser.add_argument(
        "--semantic-backend",
        choices=["sentence_transformer", "tfidf"],
        default="sentence_transformer",
    )
    parser.add_argument(
        "--semantic-model",
        default="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    )
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--language-training-max", type=int, default=6000)
    parser.add_argument("--bootstrap-replicates", type=int, default=1000)
    parser.add_argument("--force", action="store_true")
    return parser.parse_args()


def main():
    args = parse_args()
    repo_root = args.repo_root.expanduser().resolve()
    benchmark_dir = args.benchmark_dir.expanduser().resolve()
    output_dir = args.output_dir.expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = output_dir / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    manifest = load_manifest(benchmark_dir)
    print(f"Loaded {len(manifest):,} AuditStress variants.")
    print(manifest.groupby(["qa_track", "language"]).size())

    qa1_results = qa2_results = qa3_results = None
    if "qa1" in args.tracks:
        print("\nRunning QA1...")
        qa1_results = run_qa1(manifest, output_dir)
    if "qa2" in args.tracks:
        print("\nRunning QA2...")
        qa2_results = run_qa2(
            manifest,
            output_dir,
            cache_dir,
            args.semantic_backend,
            args.semantic_model,
            args.batch_size,
            args.bootstrap_replicates,
            args.force,
        )
    if "qa3" in args.tracks:
        print("\nRunning QA3...")
        qa3_results = run_qa3(
            manifest,
            repo_root,
            output_dir,
            cache_dir,
            args.language_training_max,
            args.semantic_backend,
            args.semantic_model,
            args.batch_size,
            args.force,
        )

    combined = build_combined_summary(
        qa1_results, qa2_results, qa3_results, output_dir
    )

    metadata = {
        "repo_root": str(repo_root),
        "benchmark_dir": str(benchmark_dir),
        "output_dir": str(output_dir),
        "tracks": args.tracks,
        "semantic_backend": args.semantic_backend,
        "semantic_model": args.semantic_model,
        "batch_size": args.batch_size,
        "language_training_max": args.language_training_max,
        "bootstrap_replicates": args.bootstrap_replicates,
        "variants": int(len(manifest)),
    }
    (output_dir / "auditstress_run_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )

    print(f"\nAudit results written to: {output_dir}")
    if not combined.empty:
        print("\nCombined summary:")
        print(combined.to_string(index=False))


## 2.1 Run QA1, QA2, and QA3 on the stress benchmark

In [ ]:

qa1_results = qa2_results = qa3_results = None

if RUN_AUDIT:
    AUDIT_DIR.mkdir(parents=True, exist_ok=True)
    cache_dir = AUDIT_DIR / 'cache'
    cache_dir.mkdir(parents=True, exist_ok=True)

    audit_manifest = load_manifest(BENCHMARK_DIR)
    print(f'Loaded {len(audit_manifest):,} variants.')
    display(
        audit_manifest.groupby(
            ['qa_track', 'language_label'],
            as_index=False,
        ).size().rename(columns={'size': 'variants'})
    )

    if 'qa1' in TRACKS:
        qa1_results = run_qa1(audit_manifest, AUDIT_DIR)
    if 'qa2' in TRACKS:
        qa2_results = run_qa2(
            audit_manifest,
            AUDIT_DIR,
            cache_dir,
            SEMANTIC_BACKEND,
            SEMANTIC_MODEL,
            BATCH_SIZE,
            BOOTSTRAP_REPLICATES,
            FORCE_RECOMPUTE,
        )
    if 'qa3' in TRACKS:
        qa3_results = run_qa3(
            audit_manifest,
            REPO_ROOT,
            AUDIT_DIR,
            cache_dir,
            LANGUAGE_TRAINING_MAX,
            SEMANTIC_BACKEND,
            SEMANTIC_MODEL,
            BATCH_SIZE,
            FORCE_RECOMPUTE,
        )

    combined_results = build_combined_summary(
        qa1_results,
        qa2_results,
        qa3_results,
        AUDIT_DIR,
    )

    run_metadata = {
        'metric_core_compatibility': metric_core.CORE_COMPATIBILITY,
        'metric_core_sha256': CORE_SHA256,
        'qa11_metric_core_sha256': QA11_CORE_SHA256,
        'qa11_evaluation_unit': 'RDF entry and target language',
        'qa12_minimum_recurring_occurrences': (
            QA12_MIN_RECURRING_OCCURRENCES
        ),
        'qa12_mapping_alignment': (
            'source and target triples aligned by position and '
            'components aligned by role'
        ),
        'qa12_metric_core_path': str(CORE_PATH),
        'qa12_scope': (
            'controlled harmful and naturally varying recurring-component '
            'groups; not a corpus-level estimate'
        ),
        'qa12_review_signal_interpretation': (
            'more than one retained normalized target form; contextual '
            'review signal, not an automatic error label'
        ),
        'qa12_stress_component_classes': [
            'predicate',
            'entity',
        ],
        'qa21_evaluation_unit': (
            'aligned RDF-to-text instance: RDF entry, lexicalisation ID, '
            'and target language'
        ),
        'qa21_shared_metric_core': True,
        'qa21_similarity_definition': (
            'cosine similarity of L2-normalised multilingual sentence '
            'embeddings'
        ),
        'qa21_predicate_alignment': (
            'all predicate pairs aligned by triple position; unequal counts '
            'or empty predicates yield undefined PredSim'
        ),
        'qa21_structured_predicate_rule': (
            'camelCase English predicate unchanged after N normalisation'
        ),
        'qa21_literal_atom_scope': (
            'subject, predicate, and object components of complete triple sets'
        ),
        'qa22_shared_metric_core': True,
        'qa22_calibration_percentile': 0.05,
        'qa22_calibration_stratification': (
            'exact actual target triple-set size; no global fallback'
        ),
        'qa22_textual_units': 'sentence-level units',
        'qa22_literal_atom_scope': (
            'subject, predicate, and object components of complete triple sets'
        ),
        'qa22_relative_literal_denominator_rule': (
            'English triples contain atoms and all are realised in English gold'
        ),
        'qa31_shared_metric_core': True,
        'qa31_evaluation_unit': (
            'aligned RDF-to-text instance: RDF entry, lexicalisation ID, '
            'and target language'
        ),
        'qa31_language_confidence_threshold': (
            metric_core.QA31_LANGUAGE_CONFIDENCE
        ),
        'qa31_code_switch_english_probability': (
            metric_core.QA31_CODE_SWITCH_ENGLISH_PROBABILITY
        ),
        'qa31_code_switch_window_rate': (
            metric_core.QA31_CODE_SWITCH_WINDOW_RATE
        ),
        'qa31_window_definition': (
            f'{metric_core.QA31_WINDOW_SIZE}-token windows, '
            f'stride {metric_core.QA31_WINDOW_STRIDE}, minimum '
            f'{metric_core.QA31_MINIMUM_WINDOW_TOKENS} tokens'
        ),
        'qa11_stress_lexicalisation_view': (
            'all English and target lexicalisations from each sampled entry'
        ),
        'expected_triple_count_rule': (
            'valid declared entry size; English modified-triple count when '
            'the declared value is missing or malformed'
        ),
        'generator_version': GENERATOR_VERSION,
        'repo_root': str(REPO_ROOT),
        'benchmark_dir': str(BENCHMARK_DIR),
        'audit_dir': str(AUDIT_DIR),
        'tracks': TRACKS,
        'semantic_backend': SEMANTIC_BACKEND,
        'semantic_model': SEMANTIC_MODEL,
        'batch_size': BATCH_SIZE,
        'language_training_max': LANGUAGE_TRAINING_MAX,
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'variants': int(len(audit_manifest)),
    }
    (AUDIT_DIR / 'auditstress_notebook_run_metadata.json').write_text(
        json.dumps(run_metadata, indent=2),
        encoding='utf-8',
    )
    display(combined_results)
    print('Audit written to:', AUDIT_DIR)
else:
    print('Audit execution skipped. Using existing outputs:', AUDIT_DIR)



# 3. Publication tables and figures

Every chart is exported independently as PDF and PNG. Tables are exported as
CSV and LaTeX. The principal paper figures are the overall heatmap, QA2 forest
plot, severity plots, and QA3 control-selectivity heatmap.


In [ ]:

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

LANGUAGE_ORDER = ['Spanish', 'Catalan']
CORRUPTION_LABELS = {
    'qa1_delete_triple': 'Deleted triple',
    'qa1_duplicate_triple': 'Duplicated triple',
    'qa1_missing_text': 'Missing text',
    'qa1_wrong_lid': 'Wrong lexicalisation ID',
    'qa1_duplicate_lid': 'Duplicated lexicalisation ID',
    'qa1_placeholder_component': 'Placeholder component',
    'qa1_missing_component': 'Missing RDF component',
    'qa1_source_markup': 'Residual source markup',
    'qa2_wrong_record_text': 'Wrong-record text',
    'qa2_entity_substitution_triple': 'Entity substitution in triples',
    'qa2_predicate_substitution_triple': 'Predicate substitution',
    'qa2_structured_predicate_retention': 'Structured unchanged-predicate challenge',
    'qa2_entity_substitution_text': 'Entity substitution in text',
    'qa2_literal_text_only': 'Literal changed in text',
    'qa2_literal_both': 'Literal changed in triples and text',
    'qa2_omit_one_sentence': 'Mild omission',
    'qa2_omit_to_first_sentence': 'Severe omission',
    'qa2_unsupported_addition_plausible': 'Plausible unsupported addition',
    'qa2_unsupported_addition_unrelated': 'Unrelated unsupported addition',
    'qa3_full_english_copy': 'Full English copy',
    'qa3_english_clause_insertion': 'English-clause insertion',
}
METRIC_LABELS = {
    'triple_coverage': 'Minimum triple coverage',
    'text_groundedness': 'Minimum text groundedness',
    'triple_similarity': 'Triple-set similarity',
    'text_similarity': 'Source-to-target text similarity',
    'literal_retention': 'Target triple-to-text literal retention',
    'source_target_literal_preservation': 'Source-to-target literal preservation',
    'predicate_similarity': 'Predicate-component similarity',
    'structured_predicate_review_flag': 'Structured predicate review flag',
}



def pretty_corruption(value):
    return CORRUPTION_LABELS.get(value, str(value).replace('_', ' ').title())


def pretty_metric(value):
    return METRIC_LABELS.get(value, str(value).replace('_', ' ').title())


def load_csv(stem, required=True):
    exact = AUDIT_DIR / f'{stem}.csv'
    if exact.exists():
        path = exact
    else:
        candidates = sorted(AUDIT_DIR.glob(f'{stem}*.csv'))
        if not candidates:
            if required:
                raise FileNotFoundError(f'{stem}.csv not found under {AUDIT_DIR}')
            return None
        path = candidates[0]
    frame = pd.read_csv(path)
    print(f'Loaded {path.name}: {len(frame):,} rows')
    return frame


def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(FIGURE_DIR / f'{stem}.png', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def export_table(frame, stem, caption, label, float_format='%.3f'):
    frame.to_csv(TABLE_DIR / f'{stem}.csv', index=False)
    latex = frame.to_latex(
        index=False,
        escape=True,
        float_format=float_format,
        caption=caption,
        label=label,
        position='t',
    )
    (TABLE_DIR / f'{stem}.tex').write_text(latex, encoding='utf-8')


def heatmap(matrix, title, colorbar_label, stem, vmin=None, vmax=None, fmt='.2f'):
    fig, ax = plt.subplots(figsize=(7.5, max(3.8, 0.43 * len(matrix) + 1.5)))
    image = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns)
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_title(title)
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix.iloc[row, column]
            if pd.notna(value):
                ax.text(column, row, format(value, fmt), ha='center', va='center')
    fig.colorbar(image, ax=ax, label=colorbar_label)
    save_figure(fig, stem)


In [ ]:

if RUN_VISUALISATIONS:
    if not AUDIT_DIR.exists():
        raise FileNotFoundError(
            f'Audit directory does not exist: {AUDIT_DIR}. Enable RUN_AUDIT or correct AUDIT_DIR.'
        )

    combined = load_csv('combined_detection_summary')
    qa1_structural = load_csv('qa1_structural_detection_summary')
    qa1_consistency = load_csv('qa1_consistency_selectivity_summary')
    qa2_corruption = load_csv('qa2_corruption_detection_summary')
    qa2_omission = load_csv('qa2_omission_severity_summary')
    qa2_addition = load_csv('qa2_addition_severity_summary')
    qa2_benign = load_csv('qa2_benign_variation_summary')
    qa3_corruption = load_csv('qa3_corruption_detection_summary')
    qa3_controls = load_csv('qa3_control_selectivity_summary')
    qa2_variants = load_csv('qa2_variant_metrics', required=False)
    qa3_variants = load_csv('qa3_variant_metrics', required=False)
else:
    print('Visualisation section disabled.')


## 3.1 Overall summary

In [ ]:
from pathlib import Path
from textwrap import fill

if RUN_VISUALISATIONS:
    overall = combined[
        ~combined["corruption_type"].isin(
            ["clean", "benign_alternative_reference"]
        )
    ].copy()

    # ---------------------------------------------------------
    # Aggregate stress-test table
    # ---------------------------------------------------------
    summary_rows = []

    for (dimension, language), group in overall.groupby(
        ["audit_dimension", "language_label"]
    ):
        weights = group["variants"].to_numpy(dtype=float)
        rates = group["primary_detection_rate"].to_numpy(dtype=float)

        summary_rows.append({
            "Audit dimension": dimension,
            "Language": language,
            "Controlled variants": int(group["variants"].sum()),
            "Corruption types": int(group["corruption_type"].nunique()),
            "Weighted detection rate": float(
                np.average(rates, weights=weights)
            ),
            "Minimum detection rate": float(
                group["primary_detection_rate"].min()
            ),
        })

    overall_table = (
        pd.DataFrame(summary_rows)
        .sort_values(["Audit dimension", "Language"])
    )

    display(
        overall_table.style.format({
            "Weighted detection rate": "{:.3f}",
            "Minimum detection rate": "{:.3f}",
        })
    )

    export_table(
        overall_table,
        "table_overall_audit_summary",
        "Controlled stress-test performance by audit dimension and language.",
        "tab:stress-overall-summary",
    )

    # ---------------------------------------------------------
    # Paper-readable horizontal heatmaps
    # ---------------------------------------------------------
    plot_data = overall.assign(
        corruption_label=overall["corruption_type"].map(
            pretty_corruption
        )
    )

    # Use the notebook's configured figure directory when available.
    figure_directory = Path(
        globals().get("FIGURE_DIR", ".")
    )
    figure_directory.mkdir(parents=True, exist_ok=True)

    def wrap_corruption_label(label, width=18):
        """Wrap long corruption labels without splitting words."""
        return fill(
            str(label),
            width=width,
            break_long_words=False,
            break_on_hyphens=False,
        )

    def plot_detection_heatmap(
        data,
        dimensions,
        title,
        filename,
        label_width=18,
    ):
        """
        Create one horizontal heatmap for a limited set of audit
        dimensions. The output size is close to the final LaTeX size,
        preventing the text from becoming tiny after scaling.
        """
        panel_data = data[
            data["audit_dimension"].isin(dimensions)
        ].copy()

        if panel_data.empty:
            print(
                f"No observations found for {dimensions}; "
                f"{filename} was not created."
            )
            return

        # Preserve the order in which corruption types occur in the data.
        column_order = (
            panel_data[
                ["audit_dimension", "corruption_label"]
            ]
            .drop_duplicates()
        )

        panel_matrix = panel_data.pivot_table(
            index="language_label",
            columns=["audit_dimension", "corruption_label"],
            values="primary_detection_rate",
            aggfunc="mean",
        )

        panel_matrix = panel_matrix.reindex(
            index=LANGUAGE_ORDER
        )

        ordered_columns = pd.MultiIndex.from_frame(
            column_order
        )

        panel_matrix = panel_matrix.reindex(
            columns=ordered_columns
        )

        # QA1 contains two sub-dimensions, so retain their prefixes.
        show_dimension = len(dimensions) > 1

        readable_labels = []

        for dimension, corruption in panel_matrix.columns:
            if show_dimension:
                label = f"{dimension}: {corruption}"
            else:
                label = corruption

            readable_labels.append(
                wrap_corruption_label(
                    label,
                    width=label_width,
                )
            )

        panel_matrix.columns = readable_labels
        panel_matrix.index.name = None
        panel_matrix.columns.name = None

        number_of_columns = panel_matrix.shape[1]

        # Generate the figure near its intended document width.
        # Avoid very wide figures that LaTeX must shrink aggressively.
        if number_of_columns <= 3:
            figure_height = 2.8
        elif number_of_columns <= 8:
            figure_height = 3.4
        else:
            figure_height = 4.0

        fig, ax = plt.subplots(
            figsize=(7.2, figure_height)
        )

        image = ax.imshow(
            panel_matrix.to_numpy(dtype=float),
            aspect="auto",
            vmin=0,
            vmax=1,
            cmap="viridis",
        )

        # Corruption types along the horizontal axis.
        ax.set_xticks(
            np.arange(number_of_columns)
        )
        ax.set_xticklabels(
            panel_matrix.columns,
            rotation=38,
            ha="right",
            rotation_mode="anchor",
            fontsize=8,
        )

        # Languages along the vertical axis.
        ax.set_yticks(
            np.arange(panel_matrix.shape[0])
        )
        ax.set_yticklabels(
            panel_matrix.index,
            fontsize=10,
        )

        # Detection rates inside the cells.
        annotation_size = (
            9 if number_of_columns <= 6 else 8
        )

        for row_index in range(panel_matrix.shape[0]):
            for column_index in range(number_of_columns):
                value = panel_matrix.iloc[
                    row_index,
                    column_index,
                ]

                if pd.isna(value):
                    continue

                text_colour = (
                    "white" if value < 0.55 else "black"
                )

                ax.text(
                    column_index,
                    row_index,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=annotation_size,
                    color=text_colour,
                )

        ax.set_title(
            title,
            fontsize=11,
            pad=10,
        )

        # Add visible cell boundaries.
        ax.set_xticks(
            np.arange(-0.5, number_of_columns, 1),
            minor=True,
        )
        ax.set_yticks(
            np.arange(
                -0.5,
                panel_matrix.shape[0],
                1,
            ),
            minor=True,
        )
        ax.grid(
            which="minor",
            linewidth=0.6,
            alpha=0.4,
        )
        ax.tick_params(
            which="minor",
            bottom=False,
            left=False,
        )

        colour_bar = fig.colorbar(
            image,
            ax=ax,
            orientation="vertical",
            fraction=0.035,
            pad=0.025,
        )
        colour_bar.set_label(
            "Detection rate",
            fontsize=9,
        )
        colour_bar.ax.tick_params(
            labelsize=8
        )

        # Provide sufficient room for the tilted labels.
        fig.subplots_adjust(
            left=0.10,
            right=0.91,
            top=0.84,
            bottom=0.43,
        )

        pdf_path = figure_directory / f"{filename}.pdf"
        png_path = figure_directory / f"{filename}.png"

        fig.savefig(
            pdf_path,
            bbox_inches="tight",
        )
        fig.savefig(
            png_path,
            dpi=300,
            bbox_inches="tight",
        )

        plt.show()
        plt.close(fig)

        print(f"Saved: {pdf_path}")
        print(f"Saved: {png_path}")

    # QA1.1 and QA1.2 fit together because they contain comparatively
    # few corruption types and are conceptually related.
    plot_detection_heatmap(
        data=plot_data,
        dimensions=["QA1.1", "QA1.2"],
        title="QA1: Structural and consistency corruption detection",
        filename="fig_qa1_detection_heatmap",
        label_width=17,
    )

    # QA2 receives its own figure because it contains the largest and
    # most varied set of corruptions.
    plot_detection_heatmap(
        data=plot_data,
        dimensions=["QA2"],
        title="QA2: Semantic and factual corruption detection",
        filename="fig_qa2_detection_heatmap",
        label_width=16,
    )

    # QA3 remains compact and can use larger labels and annotations.
    plot_detection_heatmap(
        data=plot_data,
        dimensions=["QA3.1"],
        title="QA3: Target-language corruption detection",
        filename="fig_qa3_detection_heatmap",
        label_width=20,
    )

## 3.2 QA1 figures

In [ ]:

if RUN_VISUALISATIONS:
    qa1_plot = qa1_structural[qa1_structural['qa_track'].eq('QA1-STRUCT')].copy()
    qa1_plot['Corruption'] = qa1_plot['corruption_type'].map(pretty_corruption)
    pivot = qa1_plot.pivot_table(
        index='Corruption', columns='language_label', values='detection_rate'
    ).reindex(columns=LANGUAGE_ORDER)
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    pivot.plot(kind='bar', ax=ax, rot=25)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Detection rate')
    ax.set_xlabel('')
    ax.set_title('QA1.1 structural-corruption detection')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(title='')
    save_figure(fig, 'fig_qa1_structural_detection')

    consistency = qa1_consistency.copy()
    consistency['Group'] = np.where(
        consistency['is_benign_group'], 'Natural variation', 'Controlled inconsistency'
    )
    consistency['Label'] = (
        consistency['Group'] + ' — ' + consistency['component_type'].str.title()
        + ' — ' + consistency['language_label']
    )
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    positions = np.arange(len(consistency))
    ax.bar(positions, consistency['weighted_dominant_mapping_rate'])
    ax.set_xticks(positions, consistency['Label'], rotation=30, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Weighted dominant mapping rate')
    ax.set_title('QA1.2 strength of mapping consistency')
    ax.grid(axis='y', alpha=0.25)
    save_figure(fig, 'fig_qa1_consistency_dominance')


## 3.3 QA2 effect sizes, heatmaps, severity, and benign controls

In [ ]:

if RUN_VISUALISATIONS:
    qa2_plot = qa2_corruption[
        ~qa2_corruption['corruption_type'].eq('qa2_unsupported_addition')
    ].copy()
    qa2_plot['Corruption'] = qa2_plot['corruption_type'].map(pretty_corruption)
    qa2_plot['Metric'] = qa2_plot['primary_metric'].map(pretty_metric)
    corruption_order = (
        qa2_plot.groupby('Corruption')['mean_delta'].mean().sort_values().index.tolist()
    )
    positions = np.arange(len(corruption_order))
    offsets = {'Spanish': -0.16, 'Catalan': 0.16}
    fig, ax = plt.subplots(figsize=(9.5, max(5.5, 0.48 * len(corruption_order) + 1.5)))
    for language in LANGUAGE_ORDER:
        subset = qa2_plot[qa2_plot['language_label'].eq(language)].set_index('Corruption').reindex(corruption_order)
        lower = (subset['mean_delta'] - subset['cluster_bootstrap_ci_low']).to_numpy()
        upper = (subset['cluster_bootstrap_ci_high'] - subset['mean_delta']).to_numpy()
        ax.errorbar(
            subset['mean_delta'], positions + offsets[language],
            xerr=np.vstack([lower, upper]), fmt='o', capsize=3, label=language,
        )
    ax.axvline(0, linewidth=1)
    ax.set_yticks(positions, corruption_order)
    ax.set_xlabel('Mean paired score change (corrupted − clean)')
    ax.set_title('QA2 paired degradation with record-cluster bootstrap intervals')
    ax.grid(axis='x', alpha=0.25)
    ax.legend(title='')
    save_figure(fig, 'fig_qa2_effect_sizes_forest')

    auroc = qa2_plot.pivot_table(
        index='Corruption', columns='language_label', values='auroc_clean_vs_variant'
    ).reindex(index=corruption_order, columns=LANGUAGE_ORDER)
    heatmap(auroc, 'QA2 clean-versus-corrupted discrimination', 'AUROC', 'fig_qa2_auroc_heatmap', vmin=0.5, vmax=1)

    direction = qa2_plot.pivot_table(
        index='Corruption', columns='language_label', values='expected_direction_rate'
    ).reindex(index=corruption_order, columns=LANGUAGE_ORDER)
    heatmap(direction, 'QA2 proportion moving in the expected direction', 'Expected-direction rate', 'fig_qa2_expected_direction_heatmap', vmin=0, vmax=1)

    qa2_table = qa2_plot[[
        'language_label', 'Corruption', 'Metric', 'severity', 'variants', 'mean_delta',
        'cluster_bootstrap_ci_low', 'cluster_bootstrap_ci_high',
        'expected_direction_rate', 'auroc_clean_vs_variant', 'auprc_clean_vs_variant',
    ]].rename(columns={
        'language_label': 'Language', 'severity': 'Severity', 'variants': 'Variants',
        'mean_delta': 'Mean delta', 'cluster_bootstrap_ci_low': 'CI low',
        'cluster_bootstrap_ci_high': 'CI high',
        'expected_direction_rate': 'Expected-direction rate',
        'auroc_clean_vs_variant': 'AUROC', 'auprc_clean_vs_variant': 'AUPRC',
    }).sort_values(['Corruption', 'Language'])
    display(qa2_table.style.format({
        'Mean delta': '{:.3f}', 'CI low': '{:.3f}', 'CI high': '{:.3f}',
        'Expected-direction rate': '{:.3f}', 'AUROC': '{:.3f}', 'AUPRC': '{:.3f}',
    }))
    export_table(
        qa2_table, 'table_qa2_corruption_results',
        'QA2 response to controlled semantic and factual corruptions.',
        'tab:stress-qa2-corruption',
    )


In [ ]:

if RUN_VISUALISATIONS:
    omission_long = qa2_omission.melt(
        id_vars=['language_label', 'paired_instances'],
        value_vars=['severe_lower_coverage_rate', 'severe_lower_text_similarity_rate'],
        var_name='Measure', value_name='Rate',
    )
    omission_long['Measure'] = omission_long['Measure'].map({
        'severe_lower_coverage_rate': 'Minimum triple coverage',
        'severe_lower_text_similarity_rate': 'Text similarity',
    })
    omission_pivot = omission_long.pivot_table(
        index='Measure', columns='language_label', values='Rate'
    ).reindex(columns=LANGUAGE_ORDER)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    omission_pivot.plot(kind='bar', ax=ax, rot=0)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Rate: severe score lower than mild')
    ax.set_xlabel('')
    ax.set_title('QA2 nested omission severity ordering')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(title='')
    save_figure(fig, 'fig_qa2_omission_severity')

    addition = qa2_addition.set_index('language_label').reindex(LANGUAGE_ORDER)
    fig, ax = plt.subplots(figsize=(6.8, 4.4))
    ax.bar(addition.index, addition['unrelated_lower_groundedness_rate'])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Rate: unrelated addition less grounded than plausible')
    ax.set_title('QA2 unsupported-addition severity ordering')
    ax.grid(axis='y', alpha=0.25)
    save_figure(fig, 'fig_qa2_addition_severity')

    benign_similarity = qa2_benign.set_index('language_label')[[
        'mean_aligned_reference_advantage', 'positive_aligned_reference_advantage_rate'
    ]].reindex(LANGUAGE_ORDER)
    benign_similarity.columns = ['Mean aligned-reference advantage', 'Positive-advantage rate']
    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    benign_similarity.plot(kind='bar', ax=ax, rot=0)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Aligned-reference statistic')
    ax.set_title('Benign alternatives: aligned-reference advantage')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(title='')
    save_figure(fig, 'fig_qa2_benign_aligned_reference')


## 3.4 QA2 instance-level histograms

In [ ]:

if RUN_VISUALISATIONS and qa2_variants is not None:
    histogram_specs = [
        ('qa2_wrong_record_text', 'delta_triple_coverage'),
        ('qa2_literal_both', 'delta_source_target_literal_preservation'),
        ('qa2_predicate_substitution_triple', 'delta_predicate_similarity'),
        ('qa2_unsupported_addition_plausible', 'delta_text_groundedness'),
        ('qa2_unsupported_addition_unrelated', 'delta_text_groundedness'),
    ]
    for corruption_type, metric_column in histogram_specs:
        subset = qa2_variants[qa2_variants['corruption_type'].eq(corruption_type)]
        if subset.empty or metric_column not in subset.columns:
            continue
        fig, ax = plt.subplots(figsize=(7.2, 4.5))
        for language in LANGUAGE_ORDER:
            values = subset.loc[
                subset['language_label'].eq(language), metric_column
            ].dropna()
            if len(values):
                ax.hist(values, bins=30, alpha=0.55, label=language)
        ax.axvline(0, linewidth=1)
        ax.set_xlabel('Paired score change (corrupted − clean)')
        ax.set_ylabel('Instances')
        ax.set_title(f'QA2 distribution: {pretty_corruption(corruption_type)}')
        ax.legend(title='')
        save_figure(fig, f'fig_qa2_hist_{corruption_type}')
elif RUN_VISUALISATIONS:
    print('qa2_variant_metrics.csv unavailable; QA2 histograms skipped.')


## 3.5 QA3 language detection and control selectivity

In [ ]:
if RUN_VISUALISATIONS:
    qa3_plot = qa3_corruption.copy()
    qa3_plot['Corruption'] = qa3_plot['corruption_type'].map(pretty_corruption)
    pivot = qa3_plot.pivot_table(
        index='Corruption', columns='language_label', values='detection_rate'
    ).reindex(columns=LANGUAGE_ORDER)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    pivot.plot(kind='bar', ax=ax, rot=20)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Detection rate')
    ax.set_xlabel('')
    ax.set_title('QA3 controlled language-corruption detection')
    ax.grid(axis='y', alpha=0.25)
    ax.legend(title='')
    save_figure(fig, 'fig_qa3_corruption_detection')

    control_metrics = [
        'hard_flag_false_positive_rate',
        'valid_target_language_rate',
        'wrong_language_rate',
        'exact_copy_rate',
        'code_switch_rate',
    ]
    labels = {
        'hard_flag_false_positive_rate': 'Hard false positive',
        'valid_target_language_rate': 'Valid target language',
        'wrong_language_rate': 'Wrong language',
        'exact_copy_rate': 'Exact English copy',
        'code_switch_rate': 'Code switching',
    }
    long = qa3_controls.melt(
        id_vars=['qa_track', 'language_label'], value_vars=control_metrics,
        var_name='Metric', value_name='Rate',
    )
    long['Metric'] = long['Metric'].map(labels)
    long['Control'] = long['qa_track'].replace({
        'CLEAN': 'Clean', 'BENIGN-VAR': 'Benign variation'
    }) + ' — ' + long['language_label']
    matrix = long.pivot_table(index='Metric', columns='Control', values='Rate')
    preferred = [
        'Clean — Spanish', 'Clean — Catalan',
        'Benign variation — Spanish', 'Benign variation — Catalan',
    ]
    matrix = matrix.reindex(columns=[c for c in preferred if c in matrix.columns])
    heatmap(
        matrix,
        'QA3 language-diagnostic false-positive rates on controls',
        'Rate',
        'fig_qa3_control_selectivity_heatmap',
        vmin=0,
        vmax=max(0.25, float(np.nanmax(matrix.to_numpy()))),
        fmt='.3f',
    )

    qa3_control_table = qa3_controls.rename(columns={
        'qa_track': 'Control', 'language_label': 'Language', 'variants': 'Variants',
        'hard_flag_false_positive_rate': 'Hard false-positive rate',
        'valid_target_language_rate': 'Valid target-language rate',
        'wrong_language_rate': 'Wrong-language rate',
        'exact_copy_rate': 'Exact-copy rate',
        'code_switch_rate': 'Code-switch rate',
    })
    qa3_control_table['Control'] = qa3_control_table['Control'].replace({
        'CLEAN': 'Clean', 'BENIGN-VAR': 'Benign variation'
    })
    display(qa3_control_table)
    export_table(
        qa3_control_table,
        'table_qa3_control_selectivity',
        'QA3 language-diagnostic activation rates on clean and benign controls.',
        'tab:stress-qa3-controls',
    )


## 3.6 QA3 target-language probability histograms

In [ ]:

if (
    RUN_VISUALISATIONS
    and qa3_variants is not None
    and 'target_language_probability' in qa3_variants.columns
):
    selected = qa3_variants[
        qa3_variants['qa_track'].isin(['CLEAN', 'QA3-LANG'])
        & (
            qa3_variants['qa_track'].eq('CLEAN')
            | qa3_variants['corruption_type'].eq('qa3_full_english_copy')
        )
    ].copy()
    selected['Condition'] = np.where(
        selected['qa_track'].eq('CLEAN'), 'Clean', 'Full English copy'
    )
    for language in LANGUAGE_ORDER:
        subset = selected[selected['language_label'].eq(language)]
        if subset.empty:
            continue
        fig, ax = plt.subplots(figsize=(7.2, 4.5))
        for condition in ['Clean', 'Full English copy']:
            values = subset.loc[
                subset['Condition'].eq(condition), 'target_language_probability'
            ].dropna()
            if len(values):
                ax.hist(values, bins=25, alpha=0.55, label=condition)
        ax.set_xlabel('Target-language probability')
        ax.set_ylabel('Instances')
        ax.set_title(f'QA3 language-confidence distribution — {language}')
        ax.legend(title='')
        save_figure(fig, f'fig_qa3_language_probability_{language.lower()}')
elif RUN_VISUALISATIONS:
    print('Instance-level target-language probabilities unavailable; histograms skipped.')


## 3.7 Output manifest and recommended paper selection

In [ ]:

if RUN_VISUALISATIONS:
    paper_manifest = {
        'metric_core_compatibility': metric_core.CORE_COMPATIBILITY,
        'metric_core_sha256': CORE_SHA256,
        'recommended_main_paper_figures': [
            'fig_overall_detection_heatmap.pdf',
            'fig_qa2_effect_sizes_forest.pdf',
            'fig_qa2_omission_severity.pdf',
            'fig_qa2_addition_severity.pdf',
            'fig_qa3_control_selectivity_heatmap.pdf',
        ],
        'recommended_main_paper_tables': [
            'table_overall_audit_summary.tex',
            'table_qa2_corruption_results.tex',
            'table_qa3_control_selectivity.tex',
        ],
        'all_pdf_figures': sorted(path.name for path in FIGURE_DIR.glob('*.pdf')),
        'all_latex_tables': sorted(path.name for path in TABLE_DIR.glob('*.tex')),
    }
    (PAPER_DIR / 'paper_output_manifest.json').write_text(
        json.dumps(paper_manifest, indent=2), encoding='utf-8'
    )
    print('Paper outputs:', PAPER_DIR)
    print('Figures:', len(paper_manifest['all_pdf_figures']))
    print('Tables:', len(paper_manifest['all_latex_tables']))
